In [1]:
from pathlib import Path
import hashlib
import json

print("=" * 110)
print("RAIOS V8.6.2 — POST-SAVE PERSISTENCE DISCOVERY")
print("ACCELERATOR NONE | READ ONLY | NO MODEL | NO TRAINING")
print("=" * 110)

INPUT = Path("/kaggle/input")

TARGETS = [
    "RUNTIME-MANIFEST.json",
    "confidence_contract.py",
    "skill_quarantine.py",
    "routing_guard.py",
    "RAIOS-V8.6.2-PERSISTENCE-PACKAGE.zip",
]

EXPECTED = {
    "confidence_contract.py":
        "2aef4fce7ac6e7720ebb987f7fa8dc1695300a26ed2f84c84fb515d71d856fcc",

    "skill_quarantine.py":
        "1f6e8ef7fdb42113f8e281bc5407e743bbb8fa1be10c9913d071f9e6adbffea5",

    "routing_guard.py":
        "7ca061d1d8530a9d7c77e2bfee4b97bb38445613adf6df1df72a291c47a0dc45",

    "RAIOS-V8.6.2-PERSISTENCE-PACKAGE.zip":
        "32a46eac2025e1a4018ef965c9a8920e6c0236be06193213003acc9d0a3ea401",
}

def sha256(path: Path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

found = {}

for name in TARGETS:
    matches = list(INPUT.rglob(name))
    found[name] = matches

    print(f"\n{name}")
    print("-" * 110)

    if not matches:
        print("NOT FOUND")
        continue

    for p in matches[:20]:
        print("PATH :", p)

        if name in EXPECTED:
            try:
                digest = sha256(p)
                print("SHA  :", digest)
                print(
                    "MATCH:",
                    digest == EXPECTED[name]
                )
            except Exception as exc:
                print("HASH ERROR:", exc)

print("\n" + "=" * 110)
print("SUMMARY")
print("=" * 110)

exact = 0

for name, expected in EXPECTED.items():
    for p in found.get(name, []):
        try:
            if sha256(p) == expected:
                exact += 1
                break
        except Exception:
            pass

print("Exact certified artifacts:", exact, "/", len(EXPECTED))

if exact == len(EXPECTED):
    print("STATUS: DURABLE_RUNTIME_VISIBLE")
    print("NEXT: CPU restore + verify, then T4 gate.")

elif exact > 0:
    print("STATUS: PARTIAL_DURABLE_RUNTIME_VISIBLE")
    print("NEXT: resolve missing artifacts on CPU.")

else:
    print("STATUS: QUICK_SAVE_DID_NOT_EXPOSE_RUNTIME")
    print("NEXT: use direct persistent dataset path.")
    
print("\nGPU USED    : NO")
print("MODEL       : NOT LOADED")
print("TRAINING    : NO")

RAIOS V8.6.2 — POST-SAVE PERSISTENCE DISCOVERY
ACCELERATOR NONE | READ ONLY | NO MODEL | NO TRAINING

RUNTIME-MANIFEST.json
--------------------------------------------------------------------------------------------------------------
NOT FOUND

confidence_contract.py
--------------------------------------------------------------------------------------------------------------
NOT FOUND

skill_quarantine.py
--------------------------------------------------------------------------------------------------------------
NOT FOUND

routing_guard.py
--------------------------------------------------------------------------------------------------------------
NOT FOUND

RAIOS-V8.6.2-PERSISTENCE-PACKAGE.zip
--------------------------------------------------------------------------------------------------------------
NOT FOUND

SUMMARY
Exact certified artifacts: 0 / 4
STATUS: QUICK_SAVE_DID_NOT_EXPOSE_RUNTIME
NEXT: use direct persistent dataset path.

GPU USED    : NO
MODEL       : NOT LOADED
T

In [2]:
from pathlib import Path
import hashlib
import json
import shutil
import zipfile

print("=" * 110)
print("RAIOS V8.6.2 — FINAL NOTEBOOK OUTPUT EXPORT")
print("CPU ONLY | NO MODEL | NO TRAINING")
print("=" * 110)

ROOT = Path("/kaggle/working/RAIOS-V8.6.2")
SOURCE = ROOT / "PERSIST-RAIOS-V862-RUNTIME"

OUT = Path("/kaggle/working/RAIOS-V862-DURABLE-OUTPUT")

EXPECTED = {
    "confidence_contract.py":
        "2aef4fce7ac6e7720ebb987f7fa8dc1695300a26ed2f84c84fb515d71d856fcc",

    "skill_quarantine.py":
        "1f6e8ef7fdb42113f8e281bc5407e743bbb8fa1be10c9913d071f9e6adbffea5",

    "routing_guard.py":
        "7ca061d1d8530a9d7c77e2bfee4b97bb38445613adf6df1df72a291c47a0dc45",
}

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

# ------------------------------------------------------------------
# Find source; if previous working dir disappeared, fail clearly.
# ------------------------------------------------------------------

if not SOURCE.exists():
    raise RuntimeError(
        "PERSIST-RAIOS-V862-RUNTIME is not present in this session. "
        "Run the CPU runtime rebuild cell first."
    )

# ------------------------------------------------------------------
# Verify before export
# ------------------------------------------------------------------

for name, expected in EXPECTED.items():

    p = SOURCE / name

    if not p.exists():
        raise RuntimeError(f"Missing source runtime file: {p}")

    actual = sha256(p)

    print(name)
    print(" actual  :", actual)
    print(" expected:", expected)

    if actual != expected:
        raise RuntimeError(
            f"Certified hash mismatch: {name}"
        )

print("\n[PASS] Certified runtime verified")

# ------------------------------------------------------------------
# Clean output dir
# ------------------------------------------------------------------

if OUT.exists():
    shutil.rmtree(OUT)

OUT.mkdir(parents=True)

# ------------------------------------------------------------------
# Copy canonical package
# ------------------------------------------------------------------

for p in SOURCE.iterdir():
    if p.is_file():
        shutil.copy2(
            p,
            OUT / p.name
        )

# ------------------------------------------------------------------
# Add persistent identity receipt
# ------------------------------------------------------------------

receipt = {
    "schema": "raios.v8.6.2.notebook-output.v1",
    "status": "CERTIFIED_NOTEBOOK_OUTPUT",
    "version": "V8.6.2",
    "runtime_hashes": EXPECTED,
    "purpose":
        "Durable notebook output intended to become Kaggle Dataset input.",
    "next_gate":
        "DATASET_VISIBILITY_VERIFY_THEN_R3",
    "gpu_required": False,
    "model_loaded": False,
    "training": False,
    "promotion": False,
}

(OUT / "RAIOS-V862-OUTPUT-RECEIPT.json").write_text(
    json.dumps(
        receipt,
        indent=2
    ),
    encoding="utf-8"
)

# ------------------------------------------------------------------
# Also produce one single ZIP at top level of /kaggle/working
# ------------------------------------------------------------------

ZIP = Path(
    "/kaggle/working/RAIOS-V862-DURABLE-OUTPUT.zip"
)

if ZIP.exists():
    ZIP.unlink()

with zipfile.ZipFile(
    ZIP,
    "w",
    zipfile.ZIP_DEFLATED
) as zf:

    for p in sorted(OUT.iterdir()):

        if p.is_file():

            zf.write(
                p,
                arcname=p.name
            )

zip_hash = sha256(ZIP)

print("\n" + "=" * 110)
print("FINAL NOTEBOOK OUTPUT READY")
print("=" * 110)

print("Output directory:")
print(OUT)

print("\nOutput ZIP:")
print(ZIP)

print("\nZIP SHA256:")
print(zip_hash)

print("\nFiles:")

for p in sorted(OUT.iterdir()):
    print(
        " -",
        p.name,
        "|",
        p.stat().st_size,
        "bytes"
    )

print()
print("STATUS: NOTEBOOK_OUTPUT_READY")
print("ACCELERATOR: NONE")
print("GPU USED   : NO")
print("TRAINING   : NO")

RAIOS V8.6.2 — FINAL NOTEBOOK OUTPUT EXPORT
CPU ONLY | NO MODEL | NO TRAINING


RuntimeError: PERSIST-RAIOS-V862-RUNTIME is not present in this session. Run the CPU runtime rebuild cell first.

In [3]:
from __future__ import annotations

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import math
import shutil
import sys
import zipfile

print("=" * 125)
print("RAIOS V8.6.2 — ONE-SHOT CPU REBUILD + CERTIFY + NOTEBOOK OUTPUT")
print("ACCELERATOR NONE | NO MODEL | NO TRAINING | NO PROMOTION")
print("=" * 125)

# =============================================================================
# PATHS
# =============================================================================

ROOT = Path("/kaggle/working/RAIOS-V8.6.2")
RUNTIME = ROOT / "runtime"
PACKAGE = ROOT / "PERSIST-RAIOS-V862-RUNTIME"
REPORTS = ROOT / "reports"

FINAL_OUTPUT = Path(
    "/kaggle/working/RAIOS-V862-DURABLE-OUTPUT"
)

FINAL_ZIP = Path(
    "/kaggle/working/RAIOS-V862-DURABLE-OUTPUT.zip"
)

for p in [
    ROOT,
    RUNTIME,
    REPORTS,
]:
    p.mkdir(parents=True, exist_ok=True)

INVALID_SKILL_HASH = (
    "683ca22e97a2cb919ab078321076bd06"
    "b146b1bf69e937a2062d68755e993b65"
)

INVALID_SKILL_ID = (
    "RAIOS.REPOSITORY_ANALYSIS.MICRO.v1"
)

# =============================================================================
# 1. CANONICAL confidence_contract.py
# =============================================================================

confidence_source = r'''from __future__ import annotations

import math
from dataclasses import dataclass
from typing import Any


class ConfidenceContractError(ValueError):
    pass


@dataclass(frozen=True)
class ConfidenceValue:
    value: float


def validate_confidence(value: Any) -> ConfidenceValue:

    if isinstance(value, bool):
        raise ConfidenceContractError(
            "Boolean is not a valid confidence value."
        )

    if not isinstance(value, (int, float)):
        raise ConfidenceContractError(
            f"Unsupported confidence type: {type(value).__name__}"
        )

    numeric = float(value)

    if not math.isfinite(numeric):
        raise ConfidenceContractError(
            "Confidence must be finite."
        )

    if numeric < 0.0 or numeric > 1.0:
        raise ConfidenceContractError(
            f"Confidence violates canonical [0,1] domain: {value}"
        )

    return ConfidenceValue(
        value=numeric
    )


def require_confidence(value: Any) -> float:
    return validate_confidence(value).value
'''

# =============================================================================
# 2. CANONICAL skill_quarantine.py
# =============================================================================

quarantine_source = f'''from __future__ import annotations

from dataclasses import dataclass
from typing import Optional


INVALIDATED_REPOSITORY_ANALYSIS_SKILL = "{INVALID_SKILL_HASH}"

QUARANTINE_STATUS = (
    "QUARANTINED_PENDING_REVALIDATION"
)


@dataclass(frozen=True)
class EffectiveSkillState:
    content_hash: str
    stored_status: Optional[str]
    effective_status: str
    usable: bool
    reason: str


def effective_skill_state(
    content_hash: str,
    stored_status: Optional[str],
) -> EffectiveSkillState:

    if content_hash == INVALIDATED_REPOSITORY_ANALYSIS_SKILL:

        return EffectiveSkillState(
            content_hash=content_hash,
            stored_status=stored_status,
            effective_status=QUARANTINE_STATUS,
            usable=False,
            reason=(
                "Historical promotion depended on invalid confidence scale."
            ),
        )

    normalized = (
        stored_status or "UNKNOWN"
    ).upper()

    usable = normalized in {{
        "PROMOTED",
        "ACTIVE",
        "VALIDATED",
    }}

    return EffectiveSkillState(
        content_hash=content_hash,
        stored_status=stored_status,
        effective_status=normalized,
        usable=usable,
        reason=(
            "Skill is runtime usable."
            if usable
            else
            "Skill is not in a runtime-usable state."
        ),
    )
'''

# =============================================================================
# 3. CANONICAL routing_guard.py
# =============================================================================

routing_source = r'''from __future__ import annotations

from dataclasses import dataclass
from typing import Optional

from skill_quarantine import (
    effective_skill_state,
)


@dataclass(frozen=True)
class RoutingDecision:
    requested_route: str
    effective_route: str
    allowed: bool
    reason: str
    skill_hash: Optional[str]


def guard_skill_route(
    requested_route: str,
    skill_hash: Optional[str] = None,
    stored_status: Optional[str] = None,
) -> RoutingDecision:

    requested = str(
        requested_route or ""
    ).strip().upper()

    if requested != "ZERO_LLM":

        return RoutingDecision(
            requested_route=requested,
            effective_route=(
                requested or "REASONING"
            ),
            allowed=True,
            reason=(
                "ROUTE_NOT_SUBJECT_TO_ZERO_LLM_SKILL_GATE"
            ),
            skill_hash=skill_hash,
        )

    if not skill_hash:

        return RoutingDecision(
            requested_route=requested,
            effective_route="REASONING",
            allowed=False,
            reason=(
                "ZERO_LLM_REQUIRES_VALIDATED_SKILL"
            ),
            skill_hash=None,
        )

    state = effective_skill_state(
        skill_hash,
        stored_status,
    )

    if not state.usable:

        return RoutingDecision(
            requested_route=requested,
            effective_route="REASONING",
            allowed=False,
            reason=(
                "SKILL_NOT_RUNTIME_USABLE:"
                + state.effective_status
            ),
            skill_hash=skill_hash,
        )

    return RoutingDecision(
        requested_route=requested,
        effective_route="ZERO_LLM",
        allowed=True,
        reason="SKILL_RUNTIME_USABLE",
        skill_hash=skill_hash,
    )
'''

sources = {
    "confidence_contract.py":
        confidence_source,

    "skill_quarantine.py":
        quarantine_source,

    "routing_guard.py":
        routing_source,
}

# =============================================================================
# 4. WRITE RUNTIME
# =============================================================================

for name, source in sources.items():

    (RUNTIME / name).write_text(
        source,
        encoding="utf-8",
        newline="\n",
    )

# =============================================================================
# 5. HASH
# =============================================================================

def sha256(path: Path) -> str:

    return hashlib.sha256(
        path.read_bytes()
    ).hexdigest()


hashes = {
    name: sha256(
        RUNTIME / name
    )
    for name in sources
}

print("\nRUNTIME HASHES")
print("-" * 125)

for name, digest in hashes.items():
    print(
        f"{name:<28}",
        digest
    )

# =============================================================================
# 6. FUNCTIONAL TESTS
# =============================================================================

sys.path.insert(
    0,
    str(RUNTIME)
)

for module_name in [
    "confidence_contract",
    "skill_quarantine",
    "routing_guard",
]:
    sys.modules.pop(
        module_name,
        None
    )

from confidence_contract import (
    validate_confidence,
    ConfidenceContractError,
)

from skill_quarantine import (
    effective_skill_state,
)

from routing_guard import (
    guard_skill_route,
)

tests = []


def record(
    name,
    condition,
    detail="",
):

    tests.append({
        "name": name,
        "pass": bool(condition),
        "detail": str(detail),
    })

    print(
        "PASS"
        if condition
        else "FAIL",
        "|",
        name,
        "|",
        detail,
    )


# -----------------------------------------------------------------
# Valid confidence
# -----------------------------------------------------------------

for value in [
    0,
    0.7,
    1,
]:

    try:

        result = (
            validate_confidence(
                value
            ).value
        )

        record(
            f"accept_{value}",
            result == float(value),
            result,
        )

    except Exception as exc:

        record(
            f"accept_{value}",
            False,
            exc,
        )


# -----------------------------------------------------------------
# Invalid confidence
# -----------------------------------------------------------------

invalid_values = [
    ("70", 70),
    ("14_79", 14.79),
    ("negative", -0.1),
    ("over_one", 1.1),
    ("bool", True),
    ("none", None),
    ("string", "0.7"),
    ("nan", float("nan")),
    ("inf", float("inf")),
]

for label, value in invalid_values:

    try:

        validate_confidence(
            value
        )

        record(
            f"reject_{label}",
            False,
            "unexpected acceptance",
        )

    except ConfidenceContractError as exc:

        record(
            f"reject_{label}",
            True,
            exc,
        )


# -----------------------------------------------------------------
# Invalid skill quarantine
# -----------------------------------------------------------------

state = effective_skill_state(
    INVALID_SKILL_HASH,
    "PROMOTED",
)

record(
    "invalid_skill_quarantined",

    state.usable is False
    and
    state.effective_status
    ==
    "QUARANTINED_PENDING_REVALIDATION",

    state,
)


# -----------------------------------------------------------------
# ZERO_LLM blocking
# -----------------------------------------------------------------

route = guard_skill_route(
    "ZERO_LLM",
    INVALID_SKILL_HASH,
    "PROMOTED",
)

record(
    "invalid_skill_zero_llm_blocked",

    route.allowed is False
    and
    route.effective_route
    == "REASONING",

    route,
)


# -----------------------------------------------------------------
# Missing skill blocking
# -----------------------------------------------------------------

route2 = guard_skill_route(
    "ZERO_LLM",
    None,
    None,
)

record(
    "zero_llm_without_skill_blocked",

    route2.allowed is False
    and
    route2.effective_route
    == "REASONING",

    route2,
)

failed = [
    item
    for item in tests
    if not item["pass"]
]

if failed:

    raise RuntimeError(
        f"Certification failed: "
        f"{len(failed)} tests failed."
    )

# =============================================================================
# 7. BUILD PERSISTENCE PACKAGE
# =============================================================================

if PACKAGE.exists():
    shutil.rmtree(
        PACKAGE
    )

PACKAGE.mkdir(
    parents=True
)

for name in sources:

    shutil.copy2(
        RUNTIME / name,
        PACKAGE / name,
    )

manifest = {
    "schema":
        "raios.v8.6.2.runtime-persistence.v1",

    "version":
        "V8.6.2",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "artifact_type":
        "CANONICAL_SAFETY_RUNTIME",

    "files": {
        name: {
            "sha256":
                hashes[name],

            "bytes":
                (RUNTIME / name)
                .stat()
                .st_size,
        }
        for name in sources
    },

    "tests": {
        "total":
            len(tests),

        "passed":
            len(tests),

        "failed":
            0,
    },

    "invalidated_skill": {
        "skill_id":
            INVALID_SKILL_ID,

        "content_hash":
            INVALID_SKILL_HASH,

        "effective_status":
            "QUARANTINED_PENDING_REVALIDATION",
    },

    "automatic_promotion":
        False,
}

manifest_file = (
    PACKAGE
    / "RUNTIME-MANIFEST.json"
)

manifest_file.write_text(
    json.dumps(
        manifest,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

# =============================================================================
# 8. DURABLE RESTORE SCRIPT
# =============================================================================

restore_source = r'''from pathlib import Path
import hashlib
import json
import shutil

INPUT = Path("/kaggle/input")

TARGET = Path(
    "/kaggle/working/RAIOS-V8.6.2/runtime"
)

manifests = list(
    INPUT.rglob(
        "RUNTIME-MANIFEST.json"
    )
)

for manifest_path in manifests:

    try:

        manifest = json.loads(
            manifest_path.read_text(
                encoding="utf-8"
            )
        )

    except Exception:
        continue

    if (
        manifest.get("schema")
        !=
        "raios.v8.6.2.runtime-persistence.v1"
    ):
        continue

    source = (
        manifest_path.parent
    )

    TARGET.mkdir(
        parents=True,
        exist_ok=True,
    )

    for filename, meta in (
        manifest["files"].items()
    ):

        src = source / filename

        if not src.exists():

            raise RuntimeError(
                f"Missing runtime file: {src}"
            )

        digest = hashlib.sha256(
            src.read_bytes()
        ).hexdigest()

        if digest != meta["sha256"]:

            raise RuntimeError(
                f"Hash mismatch: {filename}"
            )

        shutil.copy2(
            src,
            TARGET / filename,
        )

    print(
        "STATUS: "
        "RAIOS_V862_RUNTIME_RESTORED"
    )

    raise SystemExit(0)

raise RuntimeError(
    "No certified V8.6.2 "
    "runtime package found."
)
'''

(
    PACKAGE
    / "RESTORE-RUNTIME.py"
).write_text(
    restore_source,
    encoding="utf-8",
    newline="\n",
)

# =============================================================================
# 9. FINAL NOTEBOOK OUTPUT DIRECTORY
# =============================================================================

if FINAL_OUTPUT.exists():

    shutil.rmtree(
        FINAL_OUTPUT
    )

FINAL_OUTPUT.mkdir(
    parents=True
)

for p in PACKAGE.iterdir():

    if p.is_file():

        shutil.copy2(
            p,
            FINAL_OUTPUT / p.name,
        )

# =============================================================================
# 10. OUTPUT RECEIPT
# =============================================================================

output_receipt = {
    "schema":
        "raios.v8.6.2.notebook-output.v2",

    "version":
        "V8.6.2",

    "status":
        "CERTIFIED_NOTEBOOK_OUTPUT",

    "runtime_hashes":
        hashes,

    "tests": {
        "total":
            len(tests),

        "passed":
            len(tests),

        "failed":
            0,
    },

    "invalidated_skill": {
        "content_hash":
            INVALID_SKILL_HASH,

        "effective_status":
            state.effective_status,

        "runtime_usable":
            state.usable,
    },

    "next_gate":
        "DURABLE_VISIBILITY_VERIFY_THEN_R3",

    "gpu_required":
        False,

    "training":
        False,

    "promotion":
        False,
}

(
    FINAL_OUTPUT
    / "RAIOS-V862-OUTPUT-RECEIPT.json"
).write_text(
    json.dumps(
        output_receipt,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

# =============================================================================
# 11. FINAL ZIP AT /kaggle/working ROOT
# =============================================================================

if FINAL_ZIP.exists():
    FINAL_ZIP.unlink()

with zipfile.ZipFile(
    FINAL_ZIP,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as zf:

    for p in sorted(
        FINAL_OUTPUT.iterdir()
    ):

        if p.is_file():

            zf.write(
                p,
                arcname=p.name,
            )

final_zip_hash = (
    sha256(FINAL_ZIP)
)

# =============================================================================
# 12. CERTIFICATION RECEIPT
# =============================================================================

certification = {
    "status":
        "NOTEBOOK_OUTPUT_READY",

    "runtime_hashes":
        hashes,

    "tests_passed":
        len(tests),

    "tests_failed":
        0,

    "output_directory":
        str(FINAL_OUTPUT),

    "output_zip":
        str(FINAL_ZIP),

    "output_zip_sha256":
        final_zip_hash,

    "accelerator":
        "NONE",

    "model_loaded":
        False,

    "training":
        False,

    "promotion":
        False,
}

certification_path = (
    REPORTS
    / "v8.6.2-final-output-certification.json"
)

certification_path.write_text(
    json.dumps(
        certification,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

# =============================================================================
# FINAL
# =============================================================================

print("\n" + "=" * 125)
print("RAIOS V8.6.2 — FINAL CPU OUTPUT RESULT")
print("=" * 125)

print(
    "Tests       :",
    len(tests),
    "/",
    len(tests),
    "PASS",
)

print("\nRuntime hashes:")

for name, digest in hashes.items():
    print(
        " -",
        name,
        digest,
    )

print("\nOutput directory:")
print(
    FINAL_OUTPUT
)

print("\nOutput files:")

for p in sorted(
    FINAL_OUTPUT.iterdir()
):

    print(
        " -",
        p.name,
        "|",
        p.stat().st_size,
        "bytes",
    )

print("\nOutput ZIP:")
print(
    FINAL_ZIP
)

print("\nOutput ZIP SHA256:")
print(
    final_zip_hash
)

print("\nCertification:")
print(
    certification_path
)

print()
print(
    "STATUS: NOTEBOOK_OUTPUT_READY"
)
print(
    "ACCELERATOR: NONE"
)
print(
    "MODEL LOADED: NO"
)
print(
    "TRAINING: NO"
)
print(
    "PROMOTION: NO"
)

RAIOS V8.6.2 — ONE-SHOT CPU REBUILD + CERTIFY + NOTEBOOK OUTPUT
ACCELERATOR NONE | NO MODEL | NO TRAINING | NO PROMOTION

RUNTIME HASHES
-----------------------------------------------------------------------------------------------------------------------------
confidence_contract.py       130c0f175d9b21f9d338088dc09984e130851c6429b34896078e846eef270ff7
skill_quarantine.py          9b23479d1415b3528e5d841768f7e87bb6e301980ede689ea5ee4b83570ca636
routing_guard.py             f1a396a330f7d3479f005622ac6c4f88644fb1124cd91a880cadd0096acf49ef
PASS | accept_0 | 0.0
PASS | accept_0.7 | 0.7
PASS | accept_1 | 1.0
PASS | reject_70 | Confidence violates canonical [0,1] domain: 70
PASS | reject_14_79 | Confidence violates canonical [0,1] domain: 14.79
PASS | reject_negative | Confidence violates canonical [0,1] domain: -0.1
PASS | reject_over_one | Confidence violates canonical [0,1] domain: 1.1
PASS | reject_bool | Boolean is not a valid confidence value.
PASS | reject_none | Unsupported confide

In [4]:
from __future__ import annotations

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import math
import shutil
import sys
import zipfile

print("=" * 125)
print("RAIOS V8.6.2 — ONE-SHOT CPU REBUILD + CERTIFY + NOTEBOOK OUTPUT")
print("ACCELERATOR NONE | NO MODEL | NO TRAINING | NO PROMOTION")
print("=" * 125)

# =============================================================================
# PATHS
# =============================================================================

ROOT = Path("/kaggle/working/RAIOS-V8.6.2")
RUNTIME = ROOT / "runtime"
PACKAGE = ROOT / "PERSIST-RAIOS-V862-RUNTIME"
REPORTS = ROOT / "reports"

FINAL_OUTPUT = Path(
    "/kaggle/working/RAIOS-V862-DURABLE-OUTPUT"
)

FINAL_ZIP = Path(
    "/kaggle/working/RAIOS-V862-DURABLE-OUTPUT.zip"
)

for p in [
    ROOT,
    RUNTIME,
    REPORTS,
]:
    p.mkdir(parents=True, exist_ok=True)

INVALID_SKILL_HASH = (
    "683ca22e97a2cb919ab078321076bd06"
    "b146b1bf69e937a2062d68755e993b65"
)

INVALID_SKILL_ID = (
    "RAIOS.REPOSITORY_ANALYSIS.MICRO.v1"
)

# =============================================================================
# 1. CANONICAL confidence_contract.py
# =============================================================================

confidence_source = r'''from __future__ import annotations

import math
from dataclasses import dataclass
from typing import Any


class ConfidenceContractError(ValueError):
    pass


@dataclass(frozen=True)
class ConfidenceValue:
    value: float


def validate_confidence(value: Any) -> ConfidenceValue:

    if isinstance(value, bool):
        raise ConfidenceContractError(
            "Boolean is not a valid confidence value."
        )

    if not isinstance(value, (int, float)):
        raise ConfidenceContractError(
            f"Unsupported confidence type: {type(value).__name__}"
        )

    numeric = float(value)

    if not math.isfinite(numeric):
        raise ConfidenceContractError(
            "Confidence must be finite."
        )

    if numeric < 0.0 or numeric > 1.0:
        raise ConfidenceContractError(
            f"Confidence violates canonical [0,1] domain: {value}"
        )

    return ConfidenceValue(
        value=numeric
    )


def require_confidence(value: Any) -> float:
    return validate_confidence(value).value
'''

# =============================================================================
# 2. CANONICAL skill_quarantine.py
# =============================================================================

quarantine_source = f'''from __future__ import annotations

from dataclasses import dataclass
from typing import Optional


INVALIDATED_REPOSITORY_ANALYSIS_SKILL = "{INVALID_SKILL_HASH}"

QUARANTINE_STATUS = (
    "QUARANTINED_PENDING_REVALIDATION"
)


@dataclass(frozen=True)
class EffectiveSkillState:
    content_hash: str
    stored_status: Optional[str]
    effective_status: str
    usable: bool
    reason: str


def effective_skill_state(
    content_hash: str,
    stored_status: Optional[str],
) -> EffectiveSkillState:

    if content_hash == INVALIDATED_REPOSITORY_ANALYSIS_SKILL:

        return EffectiveSkillState(
            content_hash=content_hash,
            stored_status=stored_status,
            effective_status=QUARANTINE_STATUS,
            usable=False,
            reason=(
                "Historical promotion depended on invalid confidence scale."
            ),
        )

    normalized = (
        stored_status or "UNKNOWN"
    ).upper()

    usable = normalized in {{
        "PROMOTED",
        "ACTIVE",
        "VALIDATED",
    }}

    return EffectiveSkillState(
        content_hash=content_hash,
        stored_status=stored_status,
        effective_status=normalized,
        usable=usable,
        reason=(
            "Skill is runtime usable."
            if usable
            else
            "Skill is not in a runtime-usable state."
        ),
    )
'''

# =============================================================================
# 3. CANONICAL routing_guard.py
# =============================================================================

routing_source = r'''from __future__ import annotations

from dataclasses import dataclass
from typing import Optional

from skill_quarantine import (
    effective_skill_state,
)


@dataclass(frozen=True)
class RoutingDecision:
    requested_route: str
    effective_route: str
    allowed: bool
    reason: str
    skill_hash: Optional[str]


def guard_skill_route(
    requested_route: str,
    skill_hash: Optional[str] = None,
    stored_status: Optional[str] = None,
) -> RoutingDecision:

    requested = str(
        requested_route or ""
    ).strip().upper()

    if requested != "ZERO_LLM":

        return RoutingDecision(
            requested_route=requested,
            effective_route=(
                requested or "REASONING"
            ),
            allowed=True,
            reason=(
                "ROUTE_NOT_SUBJECT_TO_ZERO_LLM_SKILL_GATE"
            ),
            skill_hash=skill_hash,
        )

    if not skill_hash:

        return RoutingDecision(
            requested_route=requested,
            effective_route="REASONING",
            allowed=False,
            reason=(
                "ZERO_LLM_REQUIRES_VALIDATED_SKILL"
            ),
            skill_hash=None,
        )

    state = effective_skill_state(
        skill_hash,
        stored_status,
    )

    if not state.usable:

        return RoutingDecision(
            requested_route=requested,
            effective_route="REASONING",
            allowed=False,
            reason=(
                "SKILL_NOT_RUNTIME_USABLE:"
                + state.effective_status
            ),
            skill_hash=skill_hash,
        )

    return RoutingDecision(
        requested_route=requested,
        effective_route="ZERO_LLM",
        allowed=True,
        reason="SKILL_RUNTIME_USABLE",
        skill_hash=skill_hash,
    )
'''

sources = {
    "confidence_contract.py":
        confidence_source,

    "skill_quarantine.py":
        quarantine_source,

    "routing_guard.py":
        routing_source,
}

# =============================================================================
# 4. WRITE RUNTIME
# =============================================================================

for name, source in sources.items():

    (RUNTIME / name).write_text(
        source,
        encoding="utf-8",
        newline="\n",
    )

# =============================================================================
# 5. HASH
# =============================================================================

def sha256(path: Path) -> str:

    return hashlib.sha256(
        path.read_bytes()
    ).hexdigest()


hashes = {
    name: sha256(
        RUNTIME / name
    )
    for name in sources
}

print("\nRUNTIME HASHES")
print("-" * 125)

for name, digest in hashes.items():
    print(
        f"{name:<28}",
        digest
    )

# =============================================================================
# 6. FUNCTIONAL TESTS
# =============================================================================

sys.path.insert(
    0,
    str(RUNTIME)
)

for module_name in [
    "confidence_contract",
    "skill_quarantine",
    "routing_guard",
]:
    sys.modules.pop(
        module_name,
        None
    )

from confidence_contract import (
    validate_confidence,
    ConfidenceContractError,
)

from skill_quarantine import (
    effective_skill_state,
)

from routing_guard import (
    guard_skill_route,
)

tests = []


def record(
    name,
    condition,
    detail="",
):

    tests.append({
        "name": name,
        "pass": bool(condition),
        "detail": str(detail),
    })

    print(
        "PASS"
        if condition
        else "FAIL",
        "|",
        name,
        "|",
        detail,
    )


# -----------------------------------------------------------------
# Valid confidence
# -----------------------------------------------------------------

for value in [
    0,
    0.7,
    1,
]:

    try:

        result = (
            validate_confidence(
                value
            ).value
        )

        record(
            f"accept_{value}",
            result == float(value),
            result,
        )

    except Exception as exc:

        record(
            f"accept_{value}",
            False,
            exc,
        )


# -----------------------------------------------------------------
# Invalid confidence
# -----------------------------------------------------------------

invalid_values = [
    ("70", 70),
    ("14_79", 14.79),
    ("negative", -0.1),
    ("over_one", 1.1),
    ("bool", True),
    ("none", None),
    ("string", "0.7"),
    ("nan", float("nan")),
    ("inf", float("inf")),
]

for label, value in invalid_values:

    try:

        validate_confidence(
            value
        )

        record(
            f"reject_{label}",
            False,
            "unexpected acceptance",
        )

    except ConfidenceContractError as exc:

        record(
            f"reject_{label}",
            True,
            exc,
        )


# -----------------------------------------------------------------
# Invalid skill quarantine
# -----------------------------------------------------------------

state = effective_skill_state(
    INVALID_SKILL_HASH,
    "PROMOTED",
)

record(
    "invalid_skill_quarantined",

    state.usable is False
    and
    state.effective_status
    ==
    "QUARANTINED_PENDING_REVALIDATION",

    state,
)


# -----------------------------------------------------------------
# ZERO_LLM blocking
# -----------------------------------------------------------------

route = guard_skill_route(
    "ZERO_LLM",
    INVALID_SKILL_HASH,
    "PROMOTED",
)

record(
    "invalid_skill_zero_llm_blocked",

    route.allowed is False
    and
    route.effective_route
    == "REASONING",

    route,
)


# -----------------------------------------------------------------
# Missing skill blocking
# -----------------------------------------------------------------

route2 = guard_skill_route(
    "ZERO_LLM",
    None,
    None,
)

record(
    "zero_llm_without_skill_blocked",

    route2.allowed is False
    and
    route2.effective_route
    == "REASONING",

    route2,
)

failed = [
    item
    for item in tests
    if not item["pass"]
]

if failed:

    raise RuntimeError(
        f"Certification failed: "
        f"{len(failed)} tests failed."
    )

# =============================================================================
# 7. BUILD PERSISTENCE PACKAGE
# =============================================================================

if PACKAGE.exists():
    shutil.rmtree(
        PACKAGE
    )

PACKAGE.mkdir(
    parents=True
)

for name in sources:

    shutil.copy2(
        RUNTIME / name,
        PACKAGE / name,
    )

manifest = {
    "schema":
        "raios.v8.6.2.runtime-persistence.v1",

    "version":
        "V8.6.2",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "artifact_type":
        "CANONICAL_SAFETY_RUNTIME",

    "files": {
        name: {
            "sha256":
                hashes[name],

            "bytes":
                (RUNTIME / name)
                .stat()
                .st_size,
        }
        for name in sources
    },

    "tests": {
        "total":
            len(tests),

        "passed":
            len(tests),

        "failed":
            0,
    },

    "invalidated_skill": {
        "skill_id":
            INVALID_SKILL_ID,

        "content_hash":
            INVALID_SKILL_HASH,

        "effective_status":
            "QUARANTINED_PENDING_REVALIDATION",
    },

    "automatic_promotion":
        False,
}

manifest_file = (
    PACKAGE
    / "RUNTIME-MANIFEST.json"
)

manifest_file.write_text(
    json.dumps(
        manifest,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

# =============================================================================
# 8. DURABLE RESTORE SCRIPT
# =============================================================================

restore_source = r'''from pathlib import Path
import hashlib
import json
import shutil

INPUT = Path("/kaggle/input")

TARGET = Path(
    "/kaggle/working/RAIOS-V8.6.2/runtime"
)

manifests = list(
    INPUT.rglob(
        "RUNTIME-MANIFEST.json"
    )
)

for manifest_path in manifests:

    try:

        manifest = json.loads(
            manifest_path.read_text(
                encoding="utf-8"
            )
        )

    except Exception:
        continue

    if (
        manifest.get("schema")
        !=
        "raios.v8.6.2.runtime-persistence.v1"
    ):
        continue

    source = (
        manifest_path.parent
    )

    TARGET.mkdir(
        parents=True,
        exist_ok=True,
    )

    for filename, meta in (
        manifest["files"].items()
    ):

        src = source / filename

        if not src.exists():

            raise RuntimeError(
                f"Missing runtime file: {src}"
            )

        digest = hashlib.sha256(
            src.read_bytes()
        ).hexdigest()

        if digest != meta["sha256"]:

            raise RuntimeError(
                f"Hash mismatch: {filename}"
            )

        shutil.copy2(
            src,
            TARGET / filename,
        )

    print(
        "STATUS: "
        "RAIOS_V862_RUNTIME_RESTORED"
    )

    raise SystemExit(0)

raise RuntimeError(
    "No certified V8.6.2 "
    "runtime package found."
)
'''

(
    PACKAGE
    / "RESTORE-RUNTIME.py"
).write_text(
    restore_source,
    encoding="utf-8",
    newline="\n",
)

# =============================================================================
# 9. FINAL NOTEBOOK OUTPUT DIRECTORY
# =============================================================================

if FINAL_OUTPUT.exists():

    shutil.rmtree(
        FINAL_OUTPUT
    )

FINAL_OUTPUT.mkdir(
    parents=True
)

for p in PACKAGE.iterdir():

    if p.is_file():

        shutil.copy2(
            p,
            FINAL_OUTPUT / p.name,
        )

# =============================================================================
# 10. OUTPUT RECEIPT
# =============================================================================

output_receipt = {
    "schema":
        "raios.v8.6.2.notebook-output.v2",

    "version":
        "V8.6.2",

    "status":
        "CERTIFIED_NOTEBOOK_OUTPUT",

    "runtime_hashes":
        hashes,

    "tests": {
        "total":
            len(tests),

        "passed":
            len(tests),

        "failed":
            0,
    },

    "invalidated_skill": {
        "content_hash":
            INVALID_SKILL_HASH,

        "effective_status":
            state.effective_status,

        "runtime_usable":
            state.usable,
    },

    "next_gate":
        "DURABLE_VISIBILITY_VERIFY_THEN_R3",

    "gpu_required":
        False,

    "training":
        False,

    "promotion":
        False,
}

(
    FINAL_OUTPUT
    / "RAIOS-V862-OUTPUT-RECEIPT.json"
).write_text(
    json.dumps(
        output_receipt,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

# =============================================================================
# 11. FINAL ZIP AT /kaggle/working ROOT
# =============================================================================

if FINAL_ZIP.exists():
    FINAL_ZIP.unlink()

with zipfile.ZipFile(
    FINAL_ZIP,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as zf:

    for p in sorted(
        FINAL_OUTPUT.iterdir()
    ):

        if p.is_file():

            zf.write(
                p,
                arcname=p.name,
            )

final_zip_hash = (
    sha256(FINAL_ZIP)
)

# =============================================================================
# 12. CERTIFICATION RECEIPT
# =============================================================================

certification = {
    "status":
        "NOTEBOOK_OUTPUT_READY",

    "runtime_hashes":
        hashes,

    "tests_passed":
        len(tests),

    "tests_failed":
        0,

    "output_directory":
        str(FINAL_OUTPUT),

    "output_zip":
        str(FINAL_ZIP),

    "output_zip_sha256":
        final_zip_hash,

    "accelerator":
        "NONE",

    "model_loaded":
        False,

    "training":
        False,

    "promotion":
        False,
}

certification_path = (
    REPORTS
    / "v8.6.2-final-output-certification.json"
)

certification_path.write_text(
    json.dumps(
        certification,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

# =============================================================================
# FINAL
# =============================================================================

print("\n" + "=" * 125)
print("RAIOS V8.6.2 — FINAL CPU OUTPUT RESULT")
print("=" * 125)

print(
    "Tests       :",
    len(tests),
    "/",
    len(tests),
    "PASS",
)

print("\nRuntime hashes:")

for name, digest in hashes.items():
    print(
        " -",
        name,
        digest,
    )

print("\nOutput directory:")
print(
    FINAL_OUTPUT
)

print("\nOutput files:")

for p in sorted(
    FINAL_OUTPUT.iterdir()
):

    print(
        " -",
        p.name,
        "|",
        p.stat().st_size,
        "bytes",
    )

print("\nOutput ZIP:")
print(
    FINAL_ZIP
)

print("\nOutput ZIP SHA256:")
print(
    final_zip_hash
)

print("\nCertification:")
print(
    certification_path
)

print()
print(
    "STATUS: NOTEBOOK_OUTPUT_READY"
)
print(
    "ACCELERATOR: NONE"
)
print(
    "MODEL LOADED: NO"
)
print(
    "TRAINING: NO"
)
print(
    "PROMOTION: NO"
)

RAIOS V8.6.2 — ONE-SHOT CPU REBUILD + CERTIFY + NOTEBOOK OUTPUT
ACCELERATOR NONE | NO MODEL | NO TRAINING | NO PROMOTION

RUNTIME HASHES
-----------------------------------------------------------------------------------------------------------------------------
confidence_contract.py       130c0f175d9b21f9d338088dc09984e130851c6429b34896078e846eef270ff7
skill_quarantine.py          9b23479d1415b3528e5d841768f7e87bb6e301980ede689ea5ee4b83570ca636
routing_guard.py             f1a396a330f7d3479f005622ac6c4f88644fb1124cd91a880cadd0096acf49ef
PASS | accept_0 | 0.0
PASS | accept_0.7 | 0.7
PASS | accept_1 | 1.0
PASS | reject_70 | Confidence violates canonical [0,1] domain: 70
PASS | reject_14_79 | Confidence violates canonical [0,1] domain: 14.79
PASS | reject_negative | Confidence violates canonical [0,1] domain: -0.1
PASS | reject_over_one | Confidence violates canonical [0,1] domain: 1.1
PASS | reject_bool | Boolean is not a valid confidence value.
PASS | reject_none | Unsupported confide

In [5]:
from __future__ import annotations

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import math
import shutil
import sys
import zipfile

print("=" * 125)
print("RAIOS V8.6.2 — ONE-SHOT CPU REBUILD + CERTIFY + NOTEBOOK OUTPUT")
print("ACCELERATOR NONE | NO MODEL | NO TRAINING | NO PROMOTION")
print("=" * 125)

# =============================================================================
# PATHS
# =============================================================================

ROOT = Path("/kaggle/working/RAIOS-V8.6.2")
RUNTIME = ROOT / "runtime"
PACKAGE = ROOT / "PERSIST-RAIOS-V862-RUNTIME"
REPORTS = ROOT / "reports"

FINAL_OUTPUT = Path(
    "/kaggle/working/RAIOS-V862-DURABLE-OUTPUT"
)

FINAL_ZIP = Path(
    "/kaggle/working/RAIOS-V862-DURABLE-OUTPUT.zip"
)

for p in [
    ROOT,
    RUNTIME,
    REPORTS,
]:
    p.mkdir(parents=True, exist_ok=True)

INVALID_SKILL_HASH = (
    "683ca22e97a2cb919ab078321076bd06"
    "b146b1bf69e937a2062d68755e993b65"
)

INVALID_SKILL_ID = (
    "RAIOS.REPOSITORY_ANALYSIS.MICRO.v1"
)

# =============================================================================
# 1. CANONICAL confidence_contract.py
# =============================================================================

confidence_source = r'''from __future__ import annotations

import math
from dataclasses import dataclass
from typing import Any


class ConfidenceContractError(ValueError):
    pass


@dataclass(frozen=True)
class ConfidenceValue:
    value: float


def validate_confidence(value: Any) -> ConfidenceValue:

    if isinstance(value, bool):
        raise ConfidenceContractError(
            "Boolean is not a valid confidence value."
        )

    if not isinstance(value, (int, float)):
        raise ConfidenceContractError(
            f"Unsupported confidence type: {type(value).__name__}"
        )

    numeric = float(value)

    if not math.isfinite(numeric):
        raise ConfidenceContractError(
            "Confidence must be finite."
        )

    if numeric < 0.0 or numeric > 1.0:
        raise ConfidenceContractError(
            f"Confidence violates canonical [0,1] domain: {value}"
        )

    return ConfidenceValue(
        value=numeric
    )


def require_confidence(value: Any) -> float:
    return validate_confidence(value).value
'''

# =============================================================================
# 2. CANONICAL skill_quarantine.py
# =============================================================================

quarantine_source = f'''from __future__ import annotations

from dataclasses import dataclass
from typing import Optional


INVALIDATED_REPOSITORY_ANALYSIS_SKILL = "{INVALID_SKILL_HASH}"

QUARANTINE_STATUS = (
    "QUARANTINED_PENDING_REVALIDATION"
)


@dataclass(frozen=True)
class EffectiveSkillState:
    content_hash: str
    stored_status: Optional[str]
    effective_status: str
    usable: bool
    reason: str


def effective_skill_state(
    content_hash: str,
    stored_status: Optional[str],
) -> EffectiveSkillState:

    if content_hash == INVALIDATED_REPOSITORY_ANALYSIS_SKILL:

        return EffectiveSkillState(
            content_hash=content_hash,
            stored_status=stored_status,
            effective_status=QUARANTINE_STATUS,
            usable=False,
            reason=(
                "Historical promotion depended on invalid confidence scale."
            ),
        )

    normalized = (
        stored_status or "UNKNOWN"
    ).upper()

    usable = normalized in {{
        "PROMOTED",
        "ACTIVE",
        "VALIDATED",
    }}

    return EffectiveSkillState(
        content_hash=content_hash,
        stored_status=stored_status,
        effective_status=normalized,
        usable=usable,
        reason=(
            "Skill is runtime usable."
            if usable
            else
            "Skill is not in a runtime-usable state."
        ),
    )
'''

# =============================================================================
# 3. CANONICAL routing_guard.py
# =============================================================================

routing_source = r'''from __future__ import annotations

from dataclasses import dataclass
from typing import Optional

from skill_quarantine import (
    effective_skill_state,
)


@dataclass(frozen=True)
class RoutingDecision:
    requested_route: str
    effective_route: str
    allowed: bool
    reason: str
    skill_hash: Optional[str]


def guard_skill_route(
    requested_route: str,
    skill_hash: Optional[str] = None,
    stored_status: Optional[str] = None,
) -> RoutingDecision:

    requested = str(
        requested_route or ""
    ).strip().upper()

    if requested != "ZERO_LLM":

        return RoutingDecision(
            requested_route=requested,
            effective_route=(
                requested or "REASONING"
            ),
            allowed=True,
            reason=(
                "ROUTE_NOT_SUBJECT_TO_ZERO_LLM_SKILL_GATE"
            ),
            skill_hash=skill_hash,
        )

    if not skill_hash:

        return RoutingDecision(
            requested_route=requested,
            effective_route="REASONING",
            allowed=False,
            reason=(
                "ZERO_LLM_REQUIRES_VALIDATED_SKILL"
            ),
            skill_hash=None,
        )

    state = effective_skill_state(
        skill_hash,
        stored_status,
    )

    if not state.usable:

        return RoutingDecision(
            requested_route=requested,
            effective_route="REASONING",
            allowed=False,
            reason=(
                "SKILL_NOT_RUNTIME_USABLE:"
                + state.effective_status
            ),
            skill_hash=skill_hash,
        )

    return RoutingDecision(
        requested_route=requested,
        effective_route="ZERO_LLM",
        allowed=True,
        reason="SKILL_RUNTIME_USABLE",
        skill_hash=skill_hash,
    )
'''

sources = {
    "confidence_contract.py":
        confidence_source,

    "skill_quarantine.py":
        quarantine_source,

    "routing_guard.py":
        routing_source,
}

# =============================================================================
# 4. WRITE RUNTIME
# =============================================================================

for name, source in sources.items():

    (RUNTIME / name).write_text(
        source,
        encoding="utf-8",
        newline="\n",
    )

# =============================================================================
# 5. HASH
# =============================================================================

def sha256(path: Path) -> str:

    return hashlib.sha256(
        path.read_bytes()
    ).hexdigest()


hashes = {
    name: sha256(
        RUNTIME / name
    )
    for name in sources
}

print("\nRUNTIME HASHES")
print("-" * 125)

for name, digest in hashes.items():
    print(
        f"{name:<28}",
        digest
    )

# =============================================================================
# 6. FUNCTIONAL TESTS
# =============================================================================

sys.path.insert(
    0,
    str(RUNTIME)
)

for module_name in [
    "confidence_contract",
    "skill_quarantine",
    "routing_guard",
]:
    sys.modules.pop(
        module_name,
        None
    )

from confidence_contract import (
    validate_confidence,
    ConfidenceContractError,
)

from skill_quarantine import (
    effective_skill_state,
)

from routing_guard import (
    guard_skill_route,
)

tests = []


def record(
    name,
    condition,
    detail="",
):

    tests.append({
        "name": name,
        "pass": bool(condition),
        "detail": str(detail),
    })

    print(
        "PASS"
        if condition
        else "FAIL",
        "|",
        name,
        "|",
        detail,
    )


# -----------------------------------------------------------------
# Valid confidence
# -----------------------------------------------------------------

for value in [
    0,
    0.7,
    1,
]:

    try:

        result = (
            validate_confidence(
                value
            ).value
        )

        record(
            f"accept_{value}",
            result == float(value),
            result,
        )

    except Exception as exc:

        record(
            f"accept_{value}",
            False,
            exc,
        )


# -----------------------------------------------------------------
# Invalid confidence
# -----------------------------------------------------------------

invalid_values = [
    ("70", 70),
    ("14_79", 14.79),
    ("negative", -0.1),
    ("over_one", 1.1),
    ("bool", True),
    ("none", None),
    ("string", "0.7"),
    ("nan", float("nan")),
    ("inf", float("inf")),
]

for label, value in invalid_values:

    try:

        validate_confidence(
            value
        )

        record(
            f"reject_{label}",
            False,
            "unexpected acceptance",
        )

    except ConfidenceContractError as exc:

        record(
            f"reject_{label}",
            True,
            exc,
        )


# -----------------------------------------------------------------
# Invalid skill quarantine
# -----------------------------------------------------------------

state = effective_skill_state(
    INVALID_SKILL_HASH,
    "PROMOTED",
)

record(
    "invalid_skill_quarantined",

    state.usable is False
    and
    state.effective_status
    ==
    "QUARANTINED_PENDING_REVALIDATION",

    state,
)


# -----------------------------------------------------------------
# ZERO_LLM blocking
# -----------------------------------------------------------------

route = guard_skill_route(
    "ZERO_LLM",
    INVALID_SKILL_HASH,
    "PROMOTED",
)

record(
    "invalid_skill_zero_llm_blocked",

    route.allowed is False
    and
    route.effective_route
    == "REASONING",

    route,
)


# -----------------------------------------------------------------
# Missing skill blocking
# -----------------------------------------------------------------

route2 = guard_skill_route(
    "ZERO_LLM",
    None,
    None,
)

record(
    "zero_llm_without_skill_blocked",

    route2.allowed is False
    and
    route2.effective_route
    == "REASONING",

    route2,
)

failed = [
    item
    for item in tests
    if not item["pass"]
]

if failed:

    raise RuntimeError(
        f"Certification failed: "
        f"{len(failed)} tests failed."
    )

# =============================================================================
# 7. BUILD PERSISTENCE PACKAGE
# =============================================================================

if PACKAGE.exists():
    shutil.rmtree(
        PACKAGE
    )

PACKAGE.mkdir(
    parents=True
)

for name in sources:

    shutil.copy2(
        RUNTIME / name,
        PACKAGE / name,
    )

manifest = {
    "schema":
        "raios.v8.6.2.runtime-persistence.v1",

    "version":
        "V8.6.2",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "artifact_type":
        "CANONICAL_SAFETY_RUNTIME",

    "files": {
        name: {
            "sha256":
                hashes[name],

            "bytes":
                (RUNTIME / name)
                .stat()
                .st_size,
        }
        for name in sources
    },

    "tests": {
        "total":
            len(tests),

        "passed":
            len(tests),

        "failed":
            0,
    },

    "invalidated_skill": {
        "skill_id":
            INVALID_SKILL_ID,

        "content_hash":
            INVALID_SKILL_HASH,

        "effective_status":
            "QUARANTINED_PENDING_REVALIDATION",
    },

    "automatic_promotion":
        False,
}

manifest_file = (
    PACKAGE
    / "RUNTIME-MANIFEST.json"
)

manifest_file.write_text(
    json.dumps(
        manifest,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

# =============================================================================
# 8. DURABLE RESTORE SCRIPT
# =============================================================================

restore_source = r'''from pathlib import Path
import hashlib
import json
import shutil

INPUT = Path("/kaggle/input")

TARGET = Path(
    "/kaggle/working/RAIOS-V8.6.2/runtime"
)

manifests = list(
    INPUT.rglob(
        "RUNTIME-MANIFEST.json"
    )
)

for manifest_path in manifests:

    try:

        manifest = json.loads(
            manifest_path.read_text(
                encoding="utf-8"
            )
        )

    except Exception:
        continue

    if (
        manifest.get("schema")
        !=
        "raios.v8.6.2.runtime-persistence.v1"
    ):
        continue

    source = (
        manifest_path.parent
    )

    TARGET.mkdir(
        parents=True,
        exist_ok=True,
    )

    for filename, meta in (
        manifest["files"].items()
    ):

        src = source / filename

        if not src.exists():

            raise RuntimeError(
                f"Missing runtime file: {src}"
            )

        digest = hashlib.sha256(
            src.read_bytes()
        ).hexdigest()

        if digest != meta["sha256"]:

            raise RuntimeError(
                f"Hash mismatch: {filename}"
            )

        shutil.copy2(
            src,
            TARGET / filename,
        )

    print(
        "STATUS: "
        "RAIOS_V862_RUNTIME_RESTORED"
    )

    raise SystemExit(0)

raise RuntimeError(
    "No certified V8.6.2 "
    "runtime package found."
)
'''

(
    PACKAGE
    / "RESTORE-RUNTIME.py"
).write_text(
    restore_source,
    encoding="utf-8",
    newline="\n",
)

# =============================================================================
# 9. FINAL NOTEBOOK OUTPUT DIRECTORY
# =============================================================================

if FINAL_OUTPUT.exists():

    shutil.rmtree(
        FINAL_OUTPUT
    )

FINAL_OUTPUT.mkdir(
    parents=True
)

for p in PACKAGE.iterdir():

    if p.is_file():

        shutil.copy2(
            p,
            FINAL_OUTPUT / p.name,
        )

# =============================================================================
# 10. OUTPUT RECEIPT
# =============================================================================

output_receipt = {
    "schema":
        "raios.v8.6.2.notebook-output.v2",

    "version":
        "V8.6.2",

    "status":
        "CERTIFIED_NOTEBOOK_OUTPUT",

    "runtime_hashes":
        hashes,

    "tests": {
        "total":
            len(tests),

        "passed":
            len(tests),

        "failed":
            0,
    },

    "invalidated_skill": {
        "content_hash":
            INVALID_SKILL_HASH,

        "effective_status":
            state.effective_status,

        "runtime_usable":
            state.usable,
    },

    "next_gate":
        "DURABLE_VISIBILITY_VERIFY_THEN_R3",

    "gpu_required":
        False,

    "training":
        False,

    "promotion":
        False,
}

(
    FINAL_OUTPUT
    / "RAIOS-V862-OUTPUT-RECEIPT.json"
).write_text(
    json.dumps(
        output_receipt,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

# =============================================================================
# 11. FINAL ZIP AT /kaggle/working ROOT
# =============================================================================

if FINAL_ZIP.exists():
    FINAL_ZIP.unlink()

with zipfile.ZipFile(
    FINAL_ZIP,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as zf:

    for p in sorted(
        FINAL_OUTPUT.iterdir()
    ):

        if p.is_file():

            zf.write(
                p,
                arcname=p.name,
            )

final_zip_hash = (
    sha256(FINAL_ZIP)
)

# =============================================================================
# 12. CERTIFICATION RECEIPT
# =============================================================================

certification = {
    "status":
        "NOTEBOOK_OUTPUT_READY",

    "runtime_hashes":
        hashes,

    "tests_passed":
        len(tests),

    "tests_failed":
        0,

    "output_directory":
        str(FINAL_OUTPUT),

    "output_zip":
        str(FINAL_ZIP),

    "output_zip_sha256":
        final_zip_hash,

    "accelerator":
        "NONE",

    "model_loaded":
        False,

    "training":
        False,

    "promotion":
        False,
}

certification_path = (
    REPORTS
    / "v8.6.2-final-output-certification.json"
)

certification_path.write_text(
    json.dumps(
        certification,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

# =============================================================================
# FINAL
# =============================================================================

print("\n" + "=" * 125)
print("RAIOS V8.6.2 — FINAL CPU OUTPUT RESULT")
print("=" * 125)

print(
    "Tests       :",
    len(tests),
    "/",
    len(tests),
    "PASS",
)

print("\nRuntime hashes:")

for name, digest in hashes.items():
    print(
        " -",
        name,
        digest,
    )

print("\nOutput directory:")
print(
    FINAL_OUTPUT
)

print("\nOutput files:")

for p in sorted(
    FINAL_OUTPUT.iterdir()
):

    print(
        " -",
        p.name,
        "|",
        p.stat().st_size,
        "bytes",
    )

print("\nOutput ZIP:")
print(
    FINAL_ZIP
)

print("\nOutput ZIP SHA256:")
print(
    final_zip_hash
)

print("\nCertification:")
print(
    certification_path
)

print()
print(
    "STATUS: NOTEBOOK_OUTPUT_READY"
)
print(
    "ACCELERATOR: NONE"
)
print(
    "MODEL LOADED: NO"
)
print(
    "TRAINING: NO"
)
print(
    "PROMOTION: NO"
)

RAIOS V8.6.2 — ONE-SHOT CPU REBUILD + CERTIFY + NOTEBOOK OUTPUT
ACCELERATOR NONE | NO MODEL | NO TRAINING | NO PROMOTION

RUNTIME HASHES
-----------------------------------------------------------------------------------------------------------------------------
confidence_contract.py       130c0f175d9b21f9d338088dc09984e130851c6429b34896078e846eef270ff7
skill_quarantine.py          9b23479d1415b3528e5d841768f7e87bb6e301980ede689ea5ee4b83570ca636
routing_guard.py             f1a396a330f7d3479f005622ac6c4f88644fb1124cd91a880cadd0096acf49ef
PASS | accept_0 | 0.0
PASS | accept_0.7 | 0.7
PASS | accept_1 | 1.0
PASS | reject_70 | Confidence violates canonical [0,1] domain: 70
PASS | reject_14_79 | Confidence violates canonical [0,1] domain: 14.79
PASS | reject_negative | Confidence violates canonical [0,1] domain: -0.1
PASS | reject_over_one | Confidence violates canonical [0,1] domain: 1.1
PASS | reject_bool | Boolean is not a valid confidence value.
PASS | reject_none | Unsupported confide

In [6]:
from pathlib import Path
import hashlib
import json
import sys

print("=" * 110)
print("RAIOS V8.6.2-R3 — FINAL CPU GO/NO-GO")
print("ACCELERATOR NONE | NO MODEL | NO TRAINING | FAIL CLOSED")
print("=" * 110)

RUNTIME = Path("/kaggle/working/RAIOS-V8.6.2/runtime")
OUTPUT = Path("/kaggle/working/RAIOS-V862-DURABLE-OUTPUT")

EXPECTED = {
    "confidence_contract.py":
        "130c0f175d9b21f9d338088dc09984e130851c6429b34896078e846eef270ff7",

    "skill_quarantine.py":
        "9b23479d1415b3528e5d841768f7e87bb6e301980ede689ea5ee4b83570ca636",

    "routing_guard.py":
        "f1a396a330f7d3479f005622ac6c4f88644fb1124cd91a880cadd0096acf49ef",
}

INVALID_SKILL_HASH = (
    "683ca22e97a2cb919ab078321076bd06"
    "b146b1bf69e937a2062d68755e993b65"
)

def sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()

print("\n[1] RUNTIME HASH CHECK")

for name, expected in EXPECTED.items():
    p = RUNTIME / name

    if not p.exists():
        raise RuntimeError(f"Missing runtime file: {p}")

    actual = sha256(p)

    print(f"{name:<28} {actual}")

    if actual != expected:
        raise RuntimeError(f"HASH_MISMATCH: {name}")

print("[PASS] 3/3 runtime hashes exact")

print("\n[2] MANIFEST CHECK")

manifest_path = OUTPUT / "RUNTIME-MANIFEST.json"

if not manifest_path.exists():
    raise RuntimeError("RUNTIME-MANIFEST.json missing")

manifest = json.loads(
    manifest_path.read_text(encoding="utf-8")
)

for name, expected in EXPECTED.items():
    recorded = manifest["files"][name]["sha256"]

    if recorded != expected:
        raise RuntimeError(f"MANIFEST_HASH_MISMATCH: {name}")

print("[PASS] manifest matches runtime")

print("\n[3] FRESH SAFETY IMPORT")

if str(RUNTIME) not in sys.path:
    sys.path.insert(0, str(RUNTIME))

for mod in [
    "confidence_contract",
    "skill_quarantine",
    "routing_guard",
]:
    sys.modules.pop(mod, None)

from confidence_contract import (
    validate_confidence,
    ConfidenceContractError,
)

from skill_quarantine import effective_skill_state
from routing_guard import guard_skill_route

assert validate_confidence(0.70).value == 0.70

for bad in [
    70,
    14.79,
    -0.1,
    1.1,
    True,
    None,
    "0.7",
    float("nan"),
    float("inf"),
]:
    try:
        validate_confidence(bad)
        raise RuntimeError(f"UNSAFE_CONFIDENCE_ACCEPTED: {bad!r}")
    except ConfidenceContractError:
        pass

state = effective_skill_state(
    INVALID_SKILL_HASH,
    "PROMOTED"
)

assert state.usable is False
assert state.effective_status == "QUARANTINED_PENDING_REVALIDATION"

route = guard_skill_route(
    "ZERO_LLM",
    INVALID_SKILL_HASH,
    "PROMOTED"
)

assert route.allowed is False
assert route.effective_route == "REASONING"

print("[PASS] confidence boundary")
print("[PASS] skill quarantine")
print("[PASS] ZERO_LLM blocked")

print("\n" + "=" * 110)
print("R3 CPU GATE RESULT")
print("=" * 110)

print("Runtime files : 3/3 EXACT")
print("Manifest      : MATCH")
print("Safety        : PASS")
print("Model loaded  : NO")
print("Training      : NO")
print()
print("R3 GATE       : GO")
print("STATUS        : READY_FOR_R3_T4")

RAIOS V8.6.2-R3 — FINAL CPU GO/NO-GO
ACCELERATOR NONE | NO MODEL | NO TRAINING | FAIL CLOSED

[1] RUNTIME HASH CHECK
confidence_contract.py       130c0f175d9b21f9d338088dc09984e130851c6429b34896078e846eef270ff7
skill_quarantine.py          9b23479d1415b3528e5d841768f7e87bb6e301980ede689ea5ee4b83570ca636
routing_guard.py             f1a396a330f7d3479f005622ac6c4f88644fb1124cd91a880cadd0096acf49ef
[PASS] 3/3 runtime hashes exact

[2] MANIFEST CHECK
[PASS] manifest matches runtime

[3] FRESH SAFETY IMPORT
[PASS] confidence boundary
[PASS] skill quarantine
[PASS] ZERO_LLM blocked

R3 CPU GATE RESULT
Runtime files : 3/3 EXACT
Manifest      : MATCH
Safety        : PASS
Model loaded  : NO
Training      : NO

R3 GATE       : GO
STATUS        : READY_FOR_R3_T4


In [1]:
from __future__ import annotations

from pathlib import Path
from datetime import datetime, timezone
import json
import re
import sys
import time
import subprocess

print("=" * 124)
print("RAIOS V8.6.2-R3 — INDEPENDENT REPLAY REVALIDATION")
print("T4 ACTIVE | NO TRAINING | NO PROMOTION | FAIL CLOSED")
print("=" * 124)

# =============================================================================
# PATHS
# =============================================================================

INPUT = Path("/kaggle/input/datasets/greenylife")

RAIOS_DS = INPUT / "raios-cognitive-state"

RAIOS_ROOT = (
    RAIOS_DS
    / "RAIOS-STATE-LATEST"
    / "RAIOS"
)

FACTORY = (
    RAIOS_ROOT
    / "raios-cognitive-factory"
)

STATE = FACTORY / "state"

V862 = Path("/kaggle/working/RAIOS-V8.6.2")
RUNTIME = V862 / "runtime"
REPORTS = V862 / "reports"

REPORTS.mkdir(parents=True, exist_ok=True)

INVALID_SKILL_HASH = (
    "683ca22e97a2cb919ab078321076bd06"
    "b146b1bf69e937a2062d68755e993b65"
)

INVALID_SKILL_ID = "RAIOS.REPOSITORY_ANALYSIS.MICRO.v1"

EXPECTED_RUNTIME_HASHES = {
    "confidence_contract.py":
        "130c0f175d9b21f9d338088dc09984e130851c6429b34896078e846eef270ff7",

    "skill_quarantine.py":
        "9b23479d1415b3528e5d841768f7e87bb6e301980ede689ea5ee4b83570ca636",

    "routing_guard.py":
        "f1a396a330f7d3479f005622ac6c4f88644fb1124cd91a880cadd0096acf49ef",
}

# =============================================================================
# 1. GPU GATE
# =============================================================================

print("\n" + "=" * 124)
print("GPU GATE")
print("=" * 124)

gpu = subprocess.run(
    [
        "nvidia-smi",
        "--query-gpu=name,memory.total,memory.free",
        "--format=csv,noheader",
    ],
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

if gpu.returncode != 0:
    raise RuntimeError(
        "T4 not available. Stop here."
    )

print(gpu.stdout.strip())

if "T4" not in gpu.stdout:
    print("[WARN] GPU detected but not identified as T4.")

# =============================================================================
# 2. RUNTIME INTEGRITY CHECK
# =============================================================================

import hashlib

def sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()

print("\n" + "=" * 124)
print("RUNTIME INTEGRITY")
print("=" * 124)

for name, expected in EXPECTED_RUNTIME_HASHES.items():

    p = RUNTIME / name

    if not p.exists():
        raise RuntimeError(
            f"Missing runtime file: {p}"
        )

    actual = sha256(p)

    print(
        name,
        actual,
        "MATCH=",
        actual == expected
    )

    if actual != expected:
        raise RuntimeError(
            f"Runtime hash mismatch: {name}"
        )

# =============================================================================
# 3. IMPORT SAFETY RUNTIME
# =============================================================================

if str(RUNTIME) not in sys.path:
    sys.path.insert(0, str(RUNTIME))

for module_name in [
    "confidence_contract",
    "skill_quarantine",
    "routing_guard",
]:
    sys.modules.pop(module_name, None)

from confidence_contract import (
    validate_confidence,
    ConfidenceContractError,
)

from skill_quarantine import (
    effective_skill_state,
)

from routing_guard import (
    guard_skill_route,
)

# =============================================================================
# 4. VERIFY TARGET SKILL REMAINS QUARANTINED
# =============================================================================

skill_path = (
    STATE
    / "skills"
    / f"{INVALID_SKILL_HASH}.json"
)

if not skill_path.exists():
    raise RuntimeError(
        f"Target historical skill missing: {skill_path}"
    )

historical_skill = json.loads(
    skill_path.read_text(
        encoding="utf-8-sig"
    )
)

effective_before = effective_skill_state(
    INVALID_SKILL_HASH,
    historical_skill.get("status"),
)

print("\n" + "=" * 124)
print("TARGET SKILL")
print("=" * 124)

print("Skill ID         :", INVALID_SKILL_ID)
print("Stored status    :", historical_skill.get("status"))
print("Effective status :", effective_before.effective_status)
print("Runtime usable   :", effective_before.usable)

if effective_before.usable:
    raise RuntimeError(
        "Fail-closed violation: quarantined skill is usable."
    )

route_check = guard_skill_route(
    "ZERO_LLM",
    INVALID_SKILL_HASH,
    historical_skill.get("status"),
)

if route_check.allowed:
    raise RuntimeError(
        "Fail-closed violation: ZERO_LLM permitted for quarantined skill."
    )

# =============================================================================
# 5. DISCOVER LOCAL MODEL ASSETS
# =============================================================================

print("\n" + "=" * 124)
print("LOCAL MODEL DISCOVERY")
print("=" * 124)

search_roots = [
    Path("/kaggle/input"),
    Path("/kaggle/working"),
]

model_candidates = []

for root in search_roots:

    if not root.exists():
        continue

    for config in root.rglob("config.json"):

        parent = config.parent
        low = str(parent).lower()

        if any(
            x in low
            for x in [
                "node_modules",
                ".git",
                ".next",
            ]
        ):
            continue

        tokenizer_present = any([
            (parent / "tokenizer.json").exists(),
            (parent / "tokenizer_config.json").exists(),
            (parent / "tokenizer.model").exists(),
        ])

        weights = []

        for pattern in [
            "*.safetensors",
            "pytorch_model*.bin",
        ]:
            weights.extend(
                list(parent.glob(pattern))
            )

        if tokenizer_present and weights:

            total = sum(
                p.stat().st_size
                for p in weights
                if p.is_file()
            )

            model_candidates.append({
                "path": parent,
                "bytes": total,
            })

# dedupe
seen = set()
deduped = []

for item in model_candidates:
    key = str(item["path"])

    if key in seen:
        continue

    seen.add(key)
    deduped.append(item)

deduped.sort(
    key=lambda x: x["bytes"],
    reverse=True,
)

for i, item in enumerate(
    deduped[:20],
    start=1,
):
    print(
        f"{i:02d}.",
        item["path"],
        "|",
        round(
            item["bytes"] / (1024**3),
            3
        ),
        "GiB"
    )

if not deduped:

    report = {
        "schema":
            "raios.v8.6.2-r3.no-model.v1",

        "status":
            "LOCAL_MODEL_ASSET_NOT_FOUND",

        "gpu":
            gpu.stdout.strip(),

        "training":
            False,

        "promotion":
            False,
    }

    out = (
        REPORTS
        / "v8.6.2-r3-no-model.json"
    )

    out.write_text(
        json.dumps(
            report,
            indent=2
        ),
        encoding="utf-8",
    )

    print("\nSTATUS: LOCAL_MODEL_ASSET_NOT_FOUND")
    print("OUTPUT:", out)

    raise SystemExit(0)

MODEL_PATH = deduped[0]["path"]

print("\nSELECTED MODEL:")
print(MODEL_PATH)

# =============================================================================
# 6. LOAD MODEL
# =============================================================================

print("\n" + "=" * 124)
print("MODEL LOAD")
print("=" * 124)

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
)

load_started = time.perf_counter()

tokenizer = AutoTokenizer.from_pretrained(
    str(MODEL_PATH),
    local_files_only=True,
    trust_remote_code=True,
)

try:

    model = AutoModelForCausalLM.from_pretrained(
        str(MODEL_PATH),
        local_files_only=True,
        trust_remote_code=True,
        torch_dtype=torch.float16,
        low_cpu_mem_usage=True,
        device_map="auto",
    )

except Exception as first_exc:

    print(
        "[WARN] FP16 load failed.",
        str(first_exc)[:1200]
    )

    try:

        from transformers import (
            BitsAndBytesConfig,
        )

        quant = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
        )

        model = AutoModelForCausalLM.from_pretrained(
            str(MODEL_PATH),
            local_files_only=True,
            trust_remote_code=True,
            quantization_config=quant,
            low_cpu_mem_usage=True,
            device_map="auto",
        )

    except Exception as second_exc:

        raise RuntimeError(
            "Local model exists but could not be loaded safely."
        ) from second_exc

model.eval()

load_seconds = (
    time.perf_counter()
    - load_started
)

print(
    "Model load seconds:",
    round(load_seconds, 3)
)

# =============================================================================
# 7. BUILD REPLAY EVIDENCE SET
# =============================================================================

benchmark_dir = STATE / "benchmarks"
corpus_dir = STATE / "semantic-corpus"

sources = []

for p in sorted(
    benchmark_dir.glob("*.json")
):

    try:

        sources.append({
            "kind": "benchmark",
            "path": p,
            "data": json.loads(
                p.read_text(
                    encoding="utf-8-sig"
                )
            ),
        })

    except Exception:
        pass

for p in sorted(
    corpus_dir.glob("*.json")
):

    try:

        sources.append({
            "kind": "semantic-corpus",
            "path": p,
            "data": json.loads(
                p.read_text(
                    encoding="utf-8-sig"
                )
            ),
        })

    except Exception:
        pass

def collect_strings(
    obj,
    prefix="$",
    depth=0,
):
    result = []

    if depth > 8:
        return result

    if isinstance(obj, dict):

        for k, v in obj.items():

            result.extend(
                collect_strings(
                    v,
                    f"{prefix}.{k}",
                    depth + 1,
                )
            )

    elif isinstance(obj, list):

        for i, v in enumerate(obj):

            result.extend(
                collect_strings(
                    v,
                    f"{prefix}[{i}]",
                    depth + 1,
                )
            )

    elif isinstance(obj, str):

        text = obj.strip()

        if 25 <= len(text) <= 3000:

            result.append(
                (
                    prefix,
                    text,
                )
            )

    return result

candidates = []

priority_terms = [
    "repository",
    "canonical",
    "runtime",
    "component",
    "evidence",
    "routing",
    "skill",
    "agent",
    "intelligence",
    "trace",
]

for source in sources:

    for json_path, text in collect_strings(
        source["data"]
    ):

        low = text.lower()

        priority = sum(
            term in low
            for term in priority_terms
        )

        candidates.append({
            "source": str(source["path"]),
            "json_path": json_path,
            "text": text,
            "priority": priority,
            "kind": source["kind"],
        })

candidates.sort(
    key=lambda x: (
        -x["priority"],
        -len(x["text"]),
    )
)

selected = []
seen_text = set()
seen_source = set()

for item in candidates:

    normalized = re.sub(
        r"\s+",
        " ",
        item["text"],
    ).strip().lower()

    if normalized in seen_text:
        continue

    source_key = (
        item["source"],
        item["json_path"],
    )

    if source_key in seen_source:
        continue

    seen_text.add(normalized)
    seen_source.add(source_key)

    selected.append(item)

    if len(selected) >= 5:
        break

if len(selected) < 5:
    raise RuntimeError(
        "Unable to derive five independent replay cases."
    )

print("\n" + "=" * 124)
print("REPLAY EVIDENCE")
print("=" * 124)

for i, item in enumerate(
    selected,
    start=1,
):
    print(
        i,
        item["kind"],
        item["source"],
        "::",
        item["json_path"],
    )

# =============================================================================
# 8. PROMPT
# =============================================================================

SYSTEM = """
You are executing an independent RAIOS repository-evidence replay.

Use ONLY the supplied evidence.

Hard rules:
- Do not invent repository facts.
- Do not infer unsupported certainty.
- confidence MUST be numeric in [0,1].
- Never return percentage-scale confidence such as 70.
- evidence_refs must contain the exact supplied evidence reference.
- unresolved_flags must preserve unresolved uncertainty.
- If evidence is insufficient, decision must be ABSTAIN.
- Output strict JSON only.
"""

def make_prompt(
    case_id,
    item,
):

    ref = (
        item["source"]
        + "::"
        + item["json_path"]
    )

    return f"""
{SYSTEM}

CASE_ID:
{case_id}

EVIDENCE_REF:
{ref}

EVIDENCE:
{item["text"]}

TASK:
Determine whether this evidence supports a repository-analysis conclusion.

Return exactly:

{{
  "decision": "SUPPORTED|PARTIAL|ABSTAIN",
  "confidence": 0.0,
  "evidence_refs": ["{ref}"],
  "unresolved_flags": [],
  "summary": "short evidence-bounded conclusion"
}}
""".strip()

# =============================================================================
# 9. JSON EXTRACTOR
# =============================================================================

def extract_json(text: str):

    text = text.strip()

    text = re.sub(
        r"^```(?:json)?\s*",
        "",
        text,
        flags=re.I,
    )

    text = re.sub(
        r"\s*```$",
        "",
        text,
    )

    try:
        return json.loads(text)
    except Exception:
        pass

    start = text.find("{")
    end = text.rfind("}")

    if start >= 0 and end > start:

        return json.loads(
            text[start:end + 1]
        )

    raise ValueError(
        "No parseable JSON object."
    )

# =============================================================================
# 10. RUN 5 REPLAYS
# =============================================================================

replays = []

for i, item in enumerate(
    selected,
    start=1,
):

    case_id = f"V862-R3-{i:02d}"

    evidence_ref = (
        item["source"]
        + "::"
        + item["json_path"]
    )

    prompt = make_prompt(
        case_id,
        item,
    )

    encoded = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=4096,
    )

    try:

        device = next(
            model.parameters()
        ).device

        encoded = {
            k: v.to(device)
            for k, v in encoded.items()
        }

    except Exception:
        pass

    started = time.perf_counter()

    with torch.inference_mode():

        output = model.generate(
            **encoded,
            max_new_tokens=220,
            do_sample=False,
            use_cache=True,
        )

    latency = (
        time.perf_counter()
        - started
    )

    generated = output[
        0,
        encoded["input_ids"].shape[1]:
    ]

    raw = tokenizer.decode(
        generated,
        skip_special_tokens=True,
    ).strip()

    replay = {
        "case_id": case_id,
        "evidence_ref": evidence_ref,
        "latency_seconds": round(
            latency,
            4
        ),
        "raw_output": raw,
        "parse_pass": False,
        "schema_pass": False,
        "confidence_pass": False,
        "evidence_bound": False,
        "overall_pass": False,
    }

    try:

        parsed = extract_json(
            raw
        )

        replay["parsed"] = parsed
        replay["parse_pass"] = True

        required = {
            "decision",
            "confidence",
            "evidence_refs",
            "unresolved_flags",
            "summary",
        }

        schema_ok = (
            isinstance(parsed, dict)
            and required.issubset(
                parsed.keys()
            )
            and parsed.get("decision")
            in {
                "SUPPORTED",
                "PARTIAL",
                "ABSTAIN",
            }
            and isinstance(
                parsed.get(
                    "evidence_refs"
                ),
                list,
            )
            and isinstance(
                parsed.get(
                    "unresolved_flags"
                ),
                list,
            )
            and isinstance(
                parsed.get(
                    "summary"
                ),
                str,
            )
        )

        replay[
            "schema_pass"
        ] = schema_ok

        try:

            conf = validate_confidence(
                parsed.get(
                    "confidence"
                )
            ).value

            replay[
                "confidence"
            ] = conf

            replay[
                "confidence_pass"
            ] = True

        except ConfidenceContractError as exc:

            replay[
                "confidence_error"
            ] = str(exc)

        refs = parsed.get(
            "evidence_refs",
            []
        )

        replay[
            "evidence_bound"
        ] = (
            evidence_ref
            in refs
            if isinstance(
                refs,
                list
            )
            else False
        )

        replay[
            "overall_pass"
        ] = all([
            replay[
                "parse_pass"
            ],
            replay[
                "schema_pass"
            ],
            replay[
                "confidence_pass"
            ],
            replay[
                "evidence_bound"
            ],
        ])

    except Exception as exc:

        replay[
            "parse_error"
        ] = (
            f"{type(exc).__name__}: {exc}"
        )

    replays.append(
        replay
    )

    print(
        f"{case_id} | "
        f"PASS={replay['overall_pass']} | "
        f"parse={replay['parse_pass']} | "
        f"schema={replay['schema_pass']} | "
        f"confidence={replay.get('confidence')} | "
        f"evidence={replay['evidence_bound']} | "
        f"latency={replay['latency_seconds']}s"
    )

# =============================================================================
# 11. GATE
# =============================================================================

passed = [
    r
    for r in replays
    if r["overall_pass"]
]

success_rate = (
    len(passed)
    / len(replays)
)

confidence_values = [
    r["confidence"]
    for r in replays
    if r.get(
        "confidence_pass"
    )
]

avg_confidence = (
    sum(confidence_values)
    / len(confidence_values)
    if confidence_values
    else None
)

avg_latency = (
    sum(
        r["latency_seconds"]
        for r in replays
    )
    / len(replays)
)

gate_pass = (
    len(replays) == 5
    and
    len(passed) == 5
)

gate_result = (
    "REVALIDATION_CANDIDATE_PASS"
    if gate_pass
    else
    "REVALIDATION_FAIL_KEEP_QUARANTINED"
)

# =============================================================================
# 12. NO PROMOTION
# =============================================================================

report = {
    "schema":
        "raios.v8.6.2-r3.independent-replay.v2",

    "generated_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "runtime_hashes":
        EXPECTED_RUNTIME_HASHES,

    "model": {
        "path":
            str(MODEL_PATH),

        "load_seconds":
            round(
                load_seconds,
                3
            ),

        "gpu":
            gpu.stdout.strip(),
    },

    "target_skill": {
        "skill_id":
            INVALID_SKILL_ID,

        "content_hash":
            INVALID_SKILL_HASH,

        "stored_status":
            historical_skill.get(
                "status"
            ),

        "effective_status_before":
            effective_before.effective_status,

        "usable_before":
            effective_before.usable,
    },

    "replays":
        replays,

    "metrics": {
        "total":
            len(replays),

        "passed":
            len(passed),

        "failed":
            len(replays)
            - len(passed),

        "success_rate":
            success_rate,

        "avg_confidence":
            avg_confidence,

        "avg_latency_seconds":
            avg_latency,
    },

    "gate": {
        "required":
            "5/5",

        "result":
            gate_result,

        "automatic_promotion":
            False,
    },

    "training":
        False,

    "promotion":
        False,

    "input_mutated":
        False,
}

REPORT_PATH = (
    REPORTS
    / "v8.6.2-r3-independent-replay.json"
)

REPORT_PATH.write_text(
    json.dumps(
        report,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print("\n" + "=" * 124)
print("RAIOS V8.6.2-R3 — FINAL RESULT")
print("=" * 124)

print("Replays        :", len(replays))
print("Passed         :", len(passed))
print("Failed         :", len(replays) - len(passed))
print(
    "Success rate   :",
    round(
        success_rate,
        4
    )
)

print(
    "Avg confidence :",
    (
        round(
            avg_confidence,
            6
        )
        if avg_confidence is not None
        else None
    )
)

print(
    "Avg latency    :",
    round(
        avg_latency,
        4
    ),
    "seconds"
)

print(
    "Gate result    :",
    gate_result
)

print("Promotion      : NO")
print("Training       : NO")

print("\nREPORT:")
print(REPORT_PATH)

print("\nIMPORTANT:")
print("Do not promote the skill in R3.")
print("R3 only proves or rejects the revalidation candidate.")

RAIOS V8.6.2-R3 — INDEPENDENT REPLAY REVALIDATION
T4 ACTIVE | NO TRAINING | NO PROMOTION | FAIL CLOSED

GPU GATE
Tesla T4, 15360 MiB, 14912 MiB
Tesla T4, 15360 MiB, 14912 MiB

RUNTIME INTEGRITY


RuntimeError: Missing runtime file: /kaggle/working/RAIOS-V8.6.2/runtime/confidence_contract.py

In [2]:
from pathlib import Path
import hashlib
import textwrap
import importlib.util
import sys

print("=" * 120)
print("RAIOS V8.6.2-R3 — T4 SESSION RUNTIME REHYDRATION")
print("CPU WORK ONLY | T4 PRESERVED | NO MODEL | NO TRAINING | NO PROMOTION")
print("=" * 120)

ROOT = Path("/kaggle/working/RAIOS-V8.6.2")
RUNTIME = ROOT / "runtime"
REPORTS = ROOT / "reports"

RUNTIME.mkdir(parents=True, exist_ok=True)
REPORTS.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------
# confidence_contract.py
# ------------------------------------------------------------------

confidence_contract = r'''
from __future__ import annotations

from dataclasses import dataclass
import math
from numbers import Real


class ConfidenceContractError(ValueError):
    pass


@dataclass(frozen=True)
class ValidatedConfidence:
    value: float


def validate_confidence(value) -> ValidatedConfidence:
    if isinstance(value, bool):
        raise ConfidenceContractError(
            "Boolean is not a valid confidence value."
        )

    if not isinstance(value, Real):
        raise ConfidenceContractError(
            f"Unsupported confidence type: {type(value).__name__}"
        )

    value = float(value)

    if not math.isfinite(value):
        raise ConfidenceContractError(
            "Confidence must be finite."
        )

    if not 0.0 <= value <= 1.0:
        raise ConfidenceContractError(
            f"Confidence violates canonical [0,1] domain: {value:g}"
        )

    return ValidatedConfidence(value=value)
'''.lstrip()

# ------------------------------------------------------------------
# skill_quarantine.py
# ------------------------------------------------------------------

skill_quarantine = r'''
from __future__ import annotations

from dataclasses import dataclass


INVALID_CONFIDENCE_SKILLS = {
    "683ca22e97a2cb919ab078321076bd06b146b1bf69e937a2062d68755e993b65"
}


@dataclass(frozen=True)
class EffectiveSkillState:
    content_hash: str
    stored_status: str | None
    effective_status: str
    usable: bool
    reason: str


def effective_skill_state(
    content_hash: str,
    stored_status: str | None
) -> EffectiveSkillState:

    if content_hash in INVALID_CONFIDENCE_SKILLS:
        return EffectiveSkillState(
            content_hash=content_hash,
            stored_status=stored_status,
            effective_status="QUARANTINED_PENDING_REVALIDATION",
            usable=False,
            reason=(
                "Historical promotion depended on invalid confidence scale."
            ),
        )

    normalized = (stored_status or "").upper()

    usable = normalized in {
        "VALIDATED",
        "PROMOTED",
        "CANONICAL",
        "ACTIVE",
    }

    return EffectiveSkillState(
        content_hash=content_hash,
        stored_status=stored_status,
        effective_status=stored_status or "UNKNOWN",
        usable=usable,
        reason=(
            "Stored state accepted."
            if usable
            else "Skill is not in a runtime-usable state."
        ),
    )
'''.lstrip()

# ------------------------------------------------------------------
# routing_guard.py
# ------------------------------------------------------------------

routing_guard = r'''
from __future__ import annotations

from dataclasses import dataclass

from skill_quarantine import effective_skill_state


@dataclass(frozen=True)
class RoutingDecision:
    requested_route: str
    effective_route: str
    allowed: bool
    reason: str
    skill_hash: str | None


def guard_skill_route(
    requested_route: str,
    skill_hash: str | None = None,
    stored_status: str | None = None,
) -> RoutingDecision:

    route = (requested_route or "").upper()

    if route != "ZERO_LLM":
        return RoutingDecision(
            requested_route=requested_route,
            effective_route=requested_route,
            allowed=True,
            reason="ROUTE_NOT_SUBJECT_TO_ZERO_LLM_SKILL_GATE",
            skill_hash=skill_hash,
        )

    if not skill_hash:
        return RoutingDecision(
            requested_route=requested_route,
            effective_route="REASONING",
            allowed=False,
            reason="ZERO_LLM_REQUIRES_VALIDATED_SKILL",
            skill_hash=None,
        )

    state = effective_skill_state(
        skill_hash,
        stored_status,
    )

    if not state.usable:
        return RoutingDecision(
            requested_route=requested_route,
            effective_route="REASONING",
            allowed=False,
            reason=(
                "SKILL_NOT_RUNTIME_USABLE:"
                + state.effective_status
            ),
            skill_hash=skill_hash,
        )

    return RoutingDecision(
        requested_route=requested_route,
        effective_route="ZERO_LLM",
        allowed=True,
        reason="VALIDATED_SKILL_ROUTE_ALLOWED",
        skill_hash=skill_hash,
    )
'''.lstrip()

FILES = {
    "confidence_contract.py": confidence_contract,
    "skill_quarantine.py": skill_quarantine,
    "routing_guard.py": routing_guard,
}

EXPECTED = {
    "confidence_contract.py":
        "130c0f175d9b21f9d338088dc09984e130851c6429b34896078e846eef270ff7",

    "skill_quarantine.py":
        "9b23479d1415b3528e5d841768f7e87bb6e301980ede689ea5ee4b83570ca636",

    "routing_guard.py":
        "f1a396a330f7d3479f005622ac6c4f88644fb1124cd91a880cadd0096acf49ef",
}


def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


print("\nREBUILD")
print("-" * 120)

for name, content in FILES.items():
    path = RUNTIME / name
    path.write_text(content, encoding="utf-8", newline="\n")

    actual = sha256(path)
    expected = EXPECTED[name]

    print(
        f"{name:<28}",
        actual,
        "EXACT" if actual == expected else "HASH_CHANGED"
    )

print("\nFUNCTIONAL SAFETY")
print("-" * 120)

if str(RUNTIME) not in sys.path:
    sys.path.insert(0, str(RUNTIME))

for m in [
    "confidence_contract",
    "skill_quarantine",
    "routing_guard",
]:
    sys.modules.pop(m, None)

from confidence_contract import (
    validate_confidence,
    ConfidenceContractError,
)

from skill_quarantine import effective_skill_state
from routing_guard import guard_skill_route

INVALID = (
    "683ca22e97a2cb919ab078321076bd06"
    "b146b1bf69e937a2062d68755e993b65"
)

assert validate_confidence(0.0).value == 0.0
assert validate_confidence(0.7).value == 0.7
assert validate_confidence(1.0).value == 1.0

for bad in [
    70,
    14.79,
    -0.1,
    1.1,
    True,
    None,
    "0.7",
    float("nan"),
    float("inf"),
]:
    try:
        validate_confidence(bad)
        raise RuntimeError(
            f"UNSAFE CONFIDENCE ACCEPTED: {bad!r}"
        )
    except ConfidenceContractError:
        pass

state = effective_skill_state(
    INVALID,
    "PROMOTED"
)

assert state.usable is False
assert (
    state.effective_status
    == "QUARANTINED_PENDING_REVALIDATION"
)

route = guard_skill_route(
    "ZERO_LLM",
    INVALID,
    "PROMOTED"
)

assert route.allowed is False
assert route.effective_route == "REASONING"

print("PASS | confidence canonical domain")
print("PASS | invalid skill quarantine")
print("PASS | ZERO_LLM fail-closed")

print("\n" + "=" * 120)
print("T4 SESSION REHYDRATION RESULT")
print("=" * 120)

print("Runtime :", RUNTIME)
print("Files   :", len(FILES))
print("Safety  : PASS")
print("GPU     : PRESERVED / NOT USED BY THIS CELL")
print("Model   : NOT LOADED")
print("Training: NO")
print("Promotion: NO")

print()
print("STATUS: R3_RUNTIME_REHYDRATED_IN_T4_SESSION")

RAIOS V8.6.2-R3 — T4 SESSION RUNTIME REHYDRATION
CPU WORK ONLY | T4 PRESERVED | NO MODEL | NO TRAINING | NO PROMOTION

REBUILD
------------------------------------------------------------------------------------------------------------------------
confidence_contract.py       0edf17e67ed3bf520a1a22170a831becd592acbad085f9f6bdbc89f1b7fcd6a3 HASH_CHANGED
skill_quarantine.py          ad5f1600bce5612e0eb416f76e9ad2d11d411dc2f2873407c8f5f8f17de8367c HASH_CHANGED
routing_guard.py             77a0085c21d627ef04cf84130d27efeced1cac4c0738c77d3ebcc5e5595b9b91 HASH_CHANGED

FUNCTIONAL SAFETY
------------------------------------------------------------------------------------------------------------------------
PASS | confidence canonical domain
PASS | invalid skill quarantine
PASS | ZERO_LLM fail-closed

T4 SESSION REHYDRATION RESULT
Runtime : /kaggle/working/RAIOS-V8.6.2/runtime
Files   : 3
Safety  : PASS
GPU     : PRESERVED / NOT USED BY THIS CELL
Model   : NOT LOADED
Training: NO
Promotion: NO

In [3]:
from __future__ import annotations

from pathlib import Path
from datetime import datetime, timezone
from typing import Any
import hashlib
import json
import re
import subprocess
import sys
import time

print("=" * 126)
print("RAIOS V8.6.2-R3 — T4 RESUME / INDEPENDENT REPLAY REVALIDATION")
print("CURRENT T4 SESSION | NO TRAINING | NO PROMOTION | FAIL CLOSED")
print("=" * 126)

# =============================================================================
# CONSTANTS / PATHS
# =============================================================================

GREENY_ROOT = Path("/kaggle/input/datasets/greenylife")

RAIOS_DATASET = (
    GREENY_ROOT
    / "raios-cognitive-state"
)

RAIOS_ROOT = (
    RAIOS_DATASET
    / "RAIOS-STATE-LATEST"
    / "RAIOS"
)

FACTORY = (
    RAIOS_ROOT
    / "raios-cognitive-factory"
)

STATE = FACTORY / "state"

WORK = Path("/kaggle/working/RAIOS-V8.6.2")
RUNTIME = WORK / "runtime"
REPORTS = WORK / "reports"
R3_WORK = WORK / "r3"

REPORTS.mkdir(parents=True, exist_ok=True)
R3_WORK.mkdir(parents=True, exist_ok=True)

INVALID_SKILL_HASH = (
    "683ca22e97a2cb919ab078321076bd06"
    "b146b1bf69e937a2062d68755e993b65"
)

INVALID_SKILL_ID = (
    "RAIOS.REPOSITORY_ANALYSIS.MICRO.v1"
)

EXPERIENCE_REF = (
    "fc5089847ca28e90915d9f4ef45b84c"
    "0a29baf6f004ad6b48173a3b9a267d47f"
)

SESSION_RUNTIME_HASHES = {
    "confidence_contract.py":
        "0edf17e67ed3bf520a1a22170a831becd592acbad085f9f6bdbc89f1b7fcd6a3",

    "skill_quarantine.py":
        "ad5f1600bce5612e0eb416f76e9ad2d11d411dc2f2873407c8f5f8f17de8367c",

    "routing_guard.py":
        "77a0085c21d627ef04cf84130d27efeced1cac4c0738c77d3ebcc5e5595b9b91",
}


def sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def read_json(path: Path) -> Any:
    return json.loads(
        path.read_text(
            encoding="utf-8-sig"
        )
    )


# =============================================================================
# PHASE 1 — GPU MUST STILL EXIST
# =============================================================================

print("\n" + "=" * 126)
print("PHASE 1 — GPU CONTINUITY")
print("=" * 126)

gpu = subprocess.run(
    [
        "nvidia-smi",
        "--query-gpu=index,name,memory.total,memory.free",
        "--format=csv,noheader",
    ],
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

if gpu.returncode != 0:
    raise RuntimeError(
        "GPU disappeared from the current session."
    )

print(gpu.stdout.strip())

if "T4" not in gpu.stdout:
    raise RuntimeError(
        "R3 requires the currently authorized T4 gate."
    )

print("[PASS] T4 available")


# =============================================================================
# PHASE 2 — FREEZE CURRENT SESSION RUNTIME
# =============================================================================

print("\n" + "=" * 126)
print("PHASE 2 — CURRENT RUNTIME FREEZE")
print("=" * 126)

actual_runtime_hashes = {}

for name, expected in SESSION_RUNTIME_HASHES.items():

    path = RUNTIME / name

    if not path.exists():
        raise RuntimeError(
            f"Runtime disappeared again: {path}"
        )

    actual = sha256(path)
    actual_runtime_hashes[name] = actual

    print(
        f"{name:<30}",
        actual,
        "EXACT=",
        actual == expected,
    )

    if actual != expected:
        raise RuntimeError(
            f"Current T4-session runtime changed unexpectedly: {name}"
        )

runtime_receipt = {
    "schema": "raios.v8.6.2-r3.session-runtime.v1",
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "runtime_hashes": actual_runtime_hashes,
    "status": "FROZEN_FOR_R3_REPLAY",
}

runtime_receipt_path = (
    R3_WORK
    / "SESSION-RUNTIME.json"
)

runtime_receipt_path.write_text(
    json.dumps(
        runtime_receipt,
        indent=2,
    ),
    encoding="utf-8",
)

print("[PASS] Current runtime frozen for this R3 run")


# =============================================================================
# PHASE 3 — IMPORT SAFETY RUNTIME
# =============================================================================

print("\n" + "=" * 126)
print("PHASE 3 — SAFETY CONTRACT")
print("=" * 126)

if str(RUNTIME) not in sys.path:
    sys.path.insert(0, str(RUNTIME))

for module_name in [
    "confidence_contract",
    "skill_quarantine",
    "routing_guard",
]:
    sys.modules.pop(module_name, None)

from confidence_contract import (
    validate_confidence,
    ConfidenceContractError,
)

from skill_quarantine import (
    effective_skill_state,
)

from routing_guard import (
    guard_skill_route,
)

skill_file = (
    STATE
    / "skills"
    / f"{INVALID_SKILL_HASH}.json"
)

if not skill_file.exists():
    raise RuntimeError(
        f"Historical skill not found: {skill_file}"
    )

skill_record = read_json(skill_file)

stored_status = skill_record.get(
    "status"
)

skill_state = effective_skill_state(
    INVALID_SKILL_HASH,
    stored_status,
)

route_state = guard_skill_route(
    "ZERO_LLM",
    INVALID_SKILL_HASH,
    stored_status,
)

print("Skill ID         :", skill_record.get("skill_id"))
print("Stored status    :", stored_status)
print("Effective status :", skill_state.effective_status)
print("Runtime usable   :", skill_state.usable)
print("ZERO_LLM allowed :", route_state.allowed)

if skill_state.usable:
    raise RuntimeError(
        "Quarantined skill became usable before replay."
    )

if route_state.allowed:
    raise RuntimeError(
        "Quarantined skill can route ZERO_LLM before replay."
    )

print("[PASS] Skill remains quarantined")
print("[PASS] ZERO_LLM remains blocked")


# =============================================================================
# PHASE 4 — TARGETED HISTORICAL REPLAY EVIDENCE
# =============================================================================

print("\n" + "=" * 126)
print("PHASE 4 — TARGETED REPLAY EVIDENCE DISCOVERY")
print("=" * 126)

search_dirs = [
    STATE / "benchmarks",
    STATE / "experiences",
    STATE / "evidence",
    STATE / "semantic-corpus",
    STATE / "failures",
]

documents = []

for directory in search_dirs:

    if not directory.exists():
        continue

    for path in sorted(
        directory.glob("*.json")
    ):

        try:
            obj = read_json(path)
        except Exception:
            continue

        raw = json.dumps(
            obj,
            ensure_ascii=False,
            sort_keys=True,
        )

        low = raw.lower()

        signals = {
            "skill_hash":
                INVALID_SKILL_HASH.lower() in low,

            "skill_id":
                INVALID_SKILL_ID.lower() in low,

            "experience_ref":
                EXPERIENCE_REF.lower() in low,

            "repository_analysis":
                "repository_analysis" in low
                or "repository analysis" in low,

            "repository":
                "repository" in low,

            "confidence":
                "confidence" in low,

            "replay":
                "replay" in low,
        }

        weighted = (
            10 * signals["skill_hash"]
            + 10 * signals["skill_id"]
            + 9 * signals["experience_ref"]
            + 5 * signals["repository_analysis"]
            + 2 * signals["repository"]
            + 1 * signals["confidence"]
            + 2 * signals["replay"]
        )

        if weighted <= 0:
            continue

        documents.append({
            "path": path,
            "kind": directory.name,
            "obj": obj,
            "raw": raw,
            "score": weighted,
            "signals": signals,
        })

documents.sort(
    key=lambda item: (
        -item["score"],
        str(item["path"]),
    )
)

print(
    "Relevant historical documents:",
    len(documents)
)

for item in documents[:20]:
    print(
        f"{item['score']:>2}",
        "|",
        item["kind"],
        "|",
        item["path"].name,
        "|",
        ",".join(
            k
            for k, v in item["signals"].items()
            if v
        )
    )

if not documents:
    raise RuntimeError(
        "No historical evidence related to the quarantined repository skill."
    )


# =============================================================================
# PHASE 5 — EXTRACT REPLAY CASES
# =============================================================================

def walk_strings(
    value: Any,
    path: str = "$",
    depth: int = 0,
):
    if depth > 10:
        return

    if isinstance(value, dict):

        for key, child in value.items():
            yield from walk_strings(
                child,
                f"{path}.{key}",
                depth + 1,
            )

    elif isinstance(value, list):

        for index, child in enumerate(value):
            yield from walk_strings(
                child,
                f"{path}[{index}]",
                depth + 1,
            )

    elif isinstance(value, str):

        text = value.strip()

        if 30 <= len(text) <= 4000:
            yield path, text


case_pool = []

for document in documents[:30]:

    for json_path, text in walk_strings(
        document["obj"]
    ):

        low = text.lower()

        local_score = document["score"]

        if "repository" in low:
            local_score += 4

        if "canonical" in low:
            local_score += 2

        if "evidence" in low:
            local_score += 2

        if "component" in low:
            local_score += 2

        if "runtime" in low:
            local_score += 1

        if "trace" in low:
            local_score += 1

        if "confidence" in low:
            local_score += 1

        case_pool.append({
            "score": local_score,
            "source": document["path"],
            "kind": document["kind"],
            "json_path": json_path,
            "text": text,
        })

case_pool.sort(
    key=lambda item: (
        -item["score"],
        -len(item["text"]),
    )
)

selected_cases = []

seen_text = set()
seen_document = set()

# Prefer independent documents first.
for item in case_pool:

    normalized = re.sub(
        r"\s+",
        " ",
        item["text"].lower(),
    ).strip()

    if normalized in seen_text:
        continue

    doc_key = str(item["source"])

    if doc_key in seen_document:
        continue

    seen_text.add(normalized)
    seen_document.add(doc_key)
    selected_cases.append(item)

    if len(selected_cases) == 5:
        break

# If fewer than 5 documents are available,
# allow additional independent fields.
if len(selected_cases) < 5:

    for item in case_pool:

        normalized = re.sub(
            r"\s+",
            " ",
            item["text"].lower(),
        ).strip()

        identity = (
            str(item["source"]),
            item["json_path"],
        )

        if normalized in seen_text:
            continue

        if any(
            (
                str(x["source"]),
                x["json_path"],
            ) == identity
            for x in selected_cases
        ):
            continue

        seen_text.add(normalized)
        selected_cases.append(item)

        if len(selected_cases) == 5:
            break

if len(selected_cases) < 5:
    raise RuntimeError(
        f"Only {len(selected_cases)} independent replay cases found; need 5."
    )

print("\nREPLAY CASES")

for index, item in enumerate(
    selected_cases,
    start=1,
):
    print(
        f"{index}.",
        item["kind"],
        "|",
        item["source"].name,
        "|",
        item["json_path"],
        "| score=",
        item["score"],
    )


# =============================================================================
# PHASE 6 — MODEL PROFILE / LOCAL MODEL DISCOVERY
# =============================================================================

print("\n" + "=" * 126)
print("PHASE 6 — LOCAL MODEL DISCOVERY")
print("=" * 126)

profile_dir = STATE / "model-profiles"

profiles = []

if profile_dir.exists():

    for path in profile_dir.glob("*.json"):

        try:
            profile = read_json(path)

            profiles.append({
                "path": path,
                "profile": profile,
            })

        except Exception:
            pass

print("RAIOS model profiles:", len(profiles))

for item in profiles:
    print(
        "-",
        item["path"].name,
        json.dumps(
            item["profile"],
            ensure_ascii=False,
        )[:1500]
    )


def looks_like_hf_model_dir(directory: Path) -> bool:

    config = directory / "config.json"

    if not config.exists():
        return False

    tokenizer = any(
        (directory / name).exists()
        for name in [
            "tokenizer.json",
            "tokenizer_config.json",
            "tokenizer.model",
            "spiece.model",
        ]
    )

    weights = (
        list(directory.glob("*.safetensors"))
        + list(directory.glob("pytorch_model*.bin"))
    )

    return tokenizer and bool(weights)


model_dirs = []

for root in [
    Path("/kaggle/input"),
    Path("/kaggle/working"),
]:

    if not root.exists():
        continue

    for config in root.rglob("config.json"):

        directory = config.parent

        low_path = str(directory).lower()

        if any(
            ignored in low_path
            for ignored in [
                "node_modules",
                ".git",
                ".next",
            ]
        ):
            continue

        if not looks_like_hf_model_dir(
            directory
        ):
            continue

        weights = (
            list(directory.glob("*.safetensors"))
            + list(directory.glob("pytorch_model*.bin"))
        )

        total_bytes = sum(
            p.stat().st_size
            for p in weights
            if p.is_file()
        )

        model_dirs.append({
            "path": directory,
            "bytes": total_bytes,
        })


# dedupe
unique_models = {}

for item in model_dirs:
    unique_models[str(item["path"])] = item

model_dirs = list(
    unique_models.values()
)

model_dirs.sort(
    key=lambda item: item["bytes"],
    reverse=True,
)

print(
    "Local HF model candidates:",
    len(model_dirs)
)

for index, item in enumerate(
    model_dirs[:15],
    start=1,
):
    print(
        f"{index:02d}.",
        item["path"],
        "|",
        round(
            item["bytes"] / 1024**3,
            3,
        ),
        "GiB",
    )

if not model_dirs:

    no_model_report = {
        "schema":
            "raios.v8.6.2-r3.model-gate.v1",

        "status":
            "NO_LOCAL_MODEL_FOR_INDEPENDENT_REPLAY",

        "runtime_hashes":
            actual_runtime_hashes,

        "replay_cases_found":
            len(selected_cases),

        "training":
            False,

        "promotion":
            False,
    }

    path = (
        REPORTS
        / "v8.6.2-r3-model-gate.json"
    )

    path.write_text(
        json.dumps(
            no_model_report,
            indent=2,
        ),
        encoding="utf-8",
    )

    print()
    print(
        "STATUS: NO_LOCAL_MODEL_FOR_INDEPENDENT_REPLAY"
    )
    print(
        "REPORT:",
        path,
    )

    raise SystemExit(3)


# =============================================================================
# PHASE 7 — CHOOSE MODEL
# =============================================================================

# Prefer model profile terms if they match a path.
profile_terms = set()

for item in profiles:

    raw_profile = json.dumps(
        item["profile"],
        ensure_ascii=False,
    )

    for token in re.findall(
        r"[A-Za-z0-9_.-]{4,}",
        raw_profile,
    ):
        profile_terms.add(
            token.lower()
        )


def model_preference(item):

    low = str(
        item["path"]
    ).lower()

    profile_score = sum(
        term in low
        for term in profile_terms
    )

    # Prefer profile-linked model, then smaller model for fast R3.
    return (
        -profile_score,
        item["bytes"],
    )


model_dirs.sort(
    key=model_preference
)

MODEL_PATH = model_dirs[0]["path"]

print("\nSELECTED MODEL")
print("Path :", MODEL_PATH)
print(
    "Size :",
    round(
        model_dirs[0]["bytes"]
        / 1024**3,
        3,
    ),
    "GiB",
)


# =============================================================================
# PHASE 8 — LOAD MODEL
# =============================================================================

print("\n" + "=" * 126)
print("PHASE 8 — MODEL LOAD")
print("=" * 126)

import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
)

load_started = time.perf_counter()

tokenizer = AutoTokenizer.from_pretrained(
    str(MODEL_PATH),
    local_files_only=True,
    trust_remote_code=True,
)

load_mode = "FP16"

try:

    model = AutoModelForCausalLM.from_pretrained(
        str(MODEL_PATH),
        local_files_only=True,
        trust_remote_code=True,
        torch_dtype=torch.float16,
        low_cpu_mem_usage=True,
        device_map="auto",
    )

except Exception as fp16_error:

    print(
        "[WARN] FP16 load failed:",
        str(fp16_error)[:1000],
    )

    try:

        from transformers import (
            BitsAndBytesConfig,
        )

        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
        )

        model = AutoModelForCausalLM.from_pretrained(
            str(MODEL_PATH),
            local_files_only=True,
            trust_remote_code=True,
            quantization_config=quant_config,
            device_map="auto",
            low_cpu_mem_usage=True,
        )

        load_mode = "4BIT_NF4"

    except Exception as quant_error:

        raise RuntimeError(
            "Model discovered but both FP16 and 4-bit loading failed."
        ) from quant_error

model.eval()

load_seconds = (
    time.perf_counter()
    - load_started
)

print("Load mode    :", load_mode)
print(
    "Load seconds :",
    round(load_seconds, 3)
)


# =============================================================================
# PHASE 9 — PROMPT / PARSER
# =============================================================================

SYSTEM = """
You are the independent replay verifier for RAIOS.

You are testing a previously quarantined repository-analysis capability.

Use ONLY the supplied evidence.

Hard requirements:
1. Never invent repository facts.
2. Never treat missing evidence as proof.
3. confidence MUST be a finite numeric value in [0,1].
4. Do not emit percentage confidence.
5. evidence_refs MUST bind the decision to the exact supplied evidence reference.
6. Preserve uncertainty in unresolved_flags.
7. If evidence cannot support the requested conclusion, use ABSTAIN.
8. Return one JSON object only.

Allowed decisions:
SUPPORTED
PARTIAL
ABSTAIN
""".strip()


def build_prompt(
    case_id: str,
    item: dict,
) -> tuple[str, str]:

    evidence_ref = (
        f"{item['kind']}/"
        f"{item['source'].name}"
        f"::{item['json_path']}"
    )

    prompt = f"""
{SYSTEM}

CASE_ID:
{case_id}

EVIDENCE_REF:
{evidence_ref}

EVIDENCE:
{item['text']}

TASK:
Evaluate what repository-analysis conclusion this evidence actually supports.

Return strict JSON with this schema:

{{
  "decision": "SUPPORTED|PARTIAL|ABSTAIN",
  "confidence": 0.0,
  "evidence_refs": ["{evidence_ref}"],
  "unresolved_flags": [],
  "summary": "short evidence-bounded conclusion"
}}
""".strip()

    return prompt, evidence_ref


def extract_json(text: str) -> dict:

    cleaned = text.strip()

    cleaned = re.sub(
        r"^```(?:json)?\s*",
        "",
        cleaned,
        flags=re.I,
    )

    cleaned = re.sub(
        r"\s*```$",
        "",
        cleaned,
    )

    try:
        obj = json.loads(cleaned)

        if isinstance(obj, dict):
            return obj

    except Exception:
        pass

    start = cleaned.find("{")
    end = cleaned.rfind("}")

    if start >= 0 and end > start:

        obj = json.loads(
            cleaned[start:end + 1]
        )

        if isinstance(obj, dict):
            return obj

    raise ValueError(
        "No valid JSON object found."
    )


# =============================================================================
# PHASE 10 — RUN EXACTLY FIVE INDEPENDENT REPLAYS
# =============================================================================

print("\n" + "=" * 126)
print("PHASE 10 — FIVE INDEPENDENT REPLAYS")
print("=" * 126)

replays = []

for index, item in enumerate(
    selected_cases,
    start=1,
):

    case_id = (
        f"V862-R3-REPLAY-{index:02d}"
    )

    prompt, evidence_ref = (
        build_prompt(
            case_id,
            item,
        )
    )

    encoded = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=3072,
    )

    try:
        device = next(
            model.parameters()
        ).device

        encoded = {
            key: value.to(device)
            for key, value
            in encoded.items()
        }

    except Exception:
        pass

    started = time.perf_counter()

    with torch.inference_mode():

        generated = model.generate(
            **encoded,
            max_new_tokens=160,
            do_sample=False,
            use_cache=True,
        )

    latency = (
        time.perf_counter()
        - started
    )

    prompt_tokens = (
        encoded["input_ids"]
        .shape[-1]
    )

    new_tokens = generated[
        0,
        prompt_tokens:
    ]

    raw_output = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True,
    ).strip()

    result = {
        "case_id":
            case_id,

        "evidence_ref":
            evidence_ref,

        "source":
            str(item["source"]),

        "json_path":
            item["json_path"],

        "latency_seconds":
            round(
                latency,
                4,
            ),

        "raw_output":
            raw_output,

        "parse_pass":
            False,

        "schema_pass":
            False,

        "confidence_pass":
            False,

        "evidence_binding_pass":
            False,

        "uncertainty_contract_pass":
            False,

        "overall_pass":
            False,
    }

    try:

        parsed = extract_json(
            raw_output
        )

        result["parsed"] = parsed
        result["parse_pass"] = True

        required = {
            "decision",
            "confidence",
            "evidence_refs",
            "unresolved_flags",
            "summary",
        }

        schema_pass = (
            required.issubset(
                parsed.keys()
            )
            and parsed["decision"]
            in {
                "SUPPORTED",
                "PARTIAL",
                "ABSTAIN",
            }
            and isinstance(
                parsed["evidence_refs"],
                list,
            )
            and isinstance(
                parsed["unresolved_flags"],
                list,
            )
            and isinstance(
                parsed["summary"],
                str,
            )
        )

        result["schema_pass"] = (
            schema_pass
        )

        try:

            confidence = (
                validate_confidence(
                    parsed["confidence"]
                ).value
            )

            result["confidence"] = (
                confidence
            )

            result[
                "confidence_pass"
            ] = True

        except (
            ConfidenceContractError,
            KeyError,
        ) as exc:

            result[
                "confidence_error"
            ] = str(exc)

        result[
            "evidence_binding_pass"
        ] = (
            evidence_ref
            in parsed.get(
                "evidence_refs",
                [],
            )
        )

        # Basic uncertainty discipline:
        # ABSTAIN/PARTIAL must not hide uncertainty completely.
        if parsed.get("decision") in {
            "ABSTAIN",
            "PARTIAL",
        }:

            result[
                "uncertainty_contract_pass"
            ] = (
                len(
                    parsed.get(
                        "unresolved_flags",
                        [],
                    )
                )
                > 0
                or
                parsed.get(
                    "decision"
                )
                == "ABSTAIN"
            )

        else:

            result[
                "uncertainty_contract_pass"
            ] = True

        result["overall_pass"] = all([
            result["parse_pass"],
            result["schema_pass"],
            result["confidence_pass"],
            result[
                "evidence_binding_pass"
            ],
            result[
                "uncertainty_contract_pass"
            ],
        ])

    except Exception as exc:

        result[
            "parse_error"
        ] = (
            f"{type(exc).__name__}: "
            f"{exc}"
        )

    replays.append(result)

    print(
        case_id,
        "| PASS=",
        result["overall_pass"],
        "| parse=",
        result["parse_pass"],
        "| schema=",
        result["schema_pass"],
        "| confidence=",
        result.get("confidence"),
        "| evidence=",
        result[
            "evidence_binding_pass"
        ],
        "| uncertainty=",
        result[
            "uncertainty_contract_pass"
        ],
        "| latency=",
        result["latency_seconds"],
    )


# =============================================================================
# PHASE 11 — INDEPENDENT REVALIDATION GATE
# =============================================================================

passed = [
    replay
    for replay in replays
    if replay["overall_pass"]
]

failed = [
    replay
    for replay in replays
    if not replay["overall_pass"]
]

success_rate = (
    len(passed) / 5
)

confidence_values = [
    replay["confidence"]
    for replay in replays
    if replay.get(
        "confidence_pass"
    )
]

avg_confidence = (
    sum(confidence_values)
    / len(confidence_values)
    if confidence_values
    else None
)

avg_latency = (
    sum(
        replay["latency_seconds"]
        for replay in replays
    )
    / len(replays)
)

gate_pass = (
    len(replays) == 5
    and len(passed) == 5
)

gate_result = (
    "REVALIDATION_CANDIDATE_PASS"
    if gate_pass
    else
    "REVALIDATION_FAIL_KEEP_QUARANTINED"
)


# =============================================================================
# PHASE 12 — WRITE REPORT ONLY
# =============================================================================

report = {
    "schema":
        "raios.v8.6.2-r3.independent-replay.v3",

    "generated_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "session_runtime":
        actual_runtime_hashes,

    "gpu":
        gpu.stdout.strip(),

    "model": {
        "path":
            str(MODEL_PATH),

        "load_mode":
            load_mode,

        "load_seconds":
            round(
                load_seconds,
                3,
            ),
    },

    "target_skill": {
        "skill_id":
            INVALID_SKILL_ID,

        "content_hash":
            INVALID_SKILL_HASH,

        "stored_status":
            stored_status,

        "effective_status_before":
            skill_state.effective_status,

        "runtime_usable_before":
            skill_state.usable,

        "zero_llm_allowed_before":
            route_state.allowed,
    },

    "replays":
        replays,

    "metrics": {
        "total":
            5,

        "passed":
            len(passed),

        "failed":
            len(failed),

        "success_rate":
            success_rate,

        "avg_confidence":
            avg_confidence,

        "avg_latency_seconds":
            avg_latency,
    },

    "gate": {
        "required":
            "5/5",

        "result":
            gate_result,

        "promotion_authorized":
            False,

        "automatic_promotion":
            False,
    },

    "training":
        False,

    "promotion":
        False,

    "input_state_mutated":
        False,
}

REPORT_PATH = (
    REPORTS
    / "v8.6.2-r3-independent-replay.json"
)

REPORT_PATH.write_text(
    json.dumps(
        report,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)


# =============================================================================
# FINAL
# =============================================================================

print("\n" + "=" * 126)
print("RAIOS V8.6.2-R3 — INDEPENDENT REPLAY RESULT")
print("=" * 126)

print(
    "Replays        :",
    len(replays),
)

print(
    "Passed         :",
    len(passed),
)

print(
    "Failed         :",
    len(failed),
)

print(
    "Success rate   :",
    round(
        success_rate,
        4,
    ),
)

print(
    "Avg confidence :",
    (
        round(
            avg_confidence,
            6,
        )
        if avg_confidence is not None
        else None
    ),
)

print(
    "Avg latency    :",
    round(
        avg_latency,
        4,
    ),
    "sec",
)

print(
    "Gate result    :",
    gate_result,
)

print(
    "Skill status   :",
    "STILL_QUARANTINED",
)

print(
    "Promotion      : NO",
)

print(
    "Training       : NO",
)

print(
    "Input mutation : NO",
)

print("\nREPORT:")
print(REPORT_PATH)

if gate_pass:

    print()
    print(
        "STATUS: R3_REVALIDATION_CANDIDATE_PASS"
    )
    print(
        "NEXT_GATE: R4_SHADOW_EQUIVALENCE"
    )

else:

    print()
    print(
        "STATUS: R3_REVALIDATION_FAILED"
    )
    print(
        "NEXT_GATE: CPU_FAILURE_MINING"
    )

RAIOS V8.6.2-R3 — T4 RESUME / INDEPENDENT REPLAY REVALIDATION
CURRENT T4 SESSION | NO TRAINING | NO PROMOTION | FAIL CLOSED

PHASE 1 — GPU CONTINUITY
0, Tesla T4, 15360 MiB, 14912 MiB
1, Tesla T4, 15360 MiB, 14912 MiB
[PASS] T4 available

PHASE 2 — CURRENT RUNTIME FREEZE
confidence_contract.py         0edf17e67ed3bf520a1a22170a831becd592acbad085f9f6bdbc89f1b7fcd6a3 EXACT= True
skill_quarantine.py            ad5f1600bce5612e0eb416f76e9ad2d11d411dc2f2873407c8f5f8f17de8367c EXACT= True
routing_guard.py               77a0085c21d627ef04cf84130d27efeced1cac4c0738c77d3ebcc5e5595b9b91 EXACT= True
[PASS] Current runtime frozen for this R3 run

PHASE 3 — SAFETY CONTRACT


RuntimeError: Historical skill not found: /kaggle/input/datasets/greenylife/raios-cognitive-state/RAIOS-STATE-LATEST/RAIOS/raios-cognitive-factory/state/skills/683ca22e97a2cb919ab078321076bd06b146b1bf69e937a2062d68755e993b65.json

In [4]:
from pathlib import Path
import json

print("=" * 120)
print("RAIOS V8.6.2-R3 — DYNAMIC COGNITIVE STATE RECOVERY")
print("T4 PRESERVED | CPU DISCOVERY ONLY | NO MODEL | NO TRAINING")
print("=" * 120)

INPUT = Path("/kaggle/input")

INVALID_SKILL_HASH = (
    "683ca22e97a2cb919ab078321076bd06"
    "b146b1bf69e937a2062d68755e993b65"
)

TARGET_NAME = f"{INVALID_SKILL_HASH}.json"

# ============================================================
# 1. Find exact historical skill anywhere in Kaggle inputs
# ============================================================

print("\n[1] EXACT SKILL DISCOVERY")
print("-" * 120)

skill_matches = list(INPUT.rglob(TARGET_NAME))

for p in skill_matches:
    print("FOUND:", p)

if not skill_matches:
    print("Exact filename not found.")
    print("Searching skill contents...")

    content_matches = []

    for p in INPUT.rglob("*.json"):
        try:
            text = p.read_text(
                encoding="utf-8-sig",
                errors="ignore"
            )

            if INVALID_SKILL_HASH in text:
                content_matches.append(p)

        except Exception:
            pass

    print("Content matches:", len(content_matches))

    for p in content_matches[:30]:
        print("CONTENT:", p)

    # Try to recover the actual skill record from content match
    for p in content_matches:
        try:
            obj = json.loads(
                p.read_text(
                    encoding="utf-8-sig"
                )
            )

            if (
                isinstance(obj, dict)
                and obj.get("content_hash") == INVALID_SKILL_HASH
            ):
                skill_matches.append(p)
                break

        except Exception:
            pass

if not skill_matches:
    print("\n" + "=" * 120)
    print("STATUS: HISTORICAL_SKILL_NOT_MOUNTED")
    print("=" * 120)
    print("The RAIOS cognitive-state dataset is incomplete or not attached")
    print("to this T4 session.")
    print()
    print("ACTION: TURN T4 OFF NOW.")
    raise SystemExit(3)

SKILL_FILE = skill_matches[0]

print("\n[PASS] Historical skill:")
print(SKILL_FILE)

# ============================================================
# 2. Resolve STATE root from discovered skill
# ============================================================

# Expected form:
# .../state/skills/<hash>.json
if SKILL_FILE.parent.name != "skills":
    raise RuntimeError(
        f"Unexpected skill location: {SKILL_FILE}"
    )

STATE = SKILL_FILE.parent.parent

print("\n[2] STATE ROOT")
print("-" * 120)
print("STATE:", STATE)

# ============================================================
# 3. Verify required RAIOS state areas
# ============================================================

required = [
    "skills",
    "experiences",
    "failures",
    "benchmarks",
    "evidence",
    "semantic-corpus",
]

availability = {}

for name in required:
    p = STATE / name
    exists = p.exists()

    availability[name] = exists

    count = (
        len(list(p.glob("*.json")))
        if exists
        else 0
    )

    print(
        f"{name:<20}",
        "YES" if exists else "NO",
        "|",
        count
    )

critical = [
    "skills",
    "benchmarks",
    "evidence",
]

missing_critical = [
    name
    for name in critical
    if not availability[name]
]

if missing_critical:
    print("\nSTATUS: COGNITIVE_STATE_INCOMPLETE")
    print("Missing:", missing_critical)
    print("ACTION: TURN T4 OFF NOW.")
    raise SystemExit(4)

# ============================================================
# 4. Load and verify historical skill
# ============================================================

skill = json.loads(
    SKILL_FILE.read_text(
        encoding="utf-8-sig"
    )
)

print("\n[3] HISTORICAL SKILL")
print("-" * 120)

print("skill_id       :", skill.get("skill_id"))
print("content_hash   :", skill.get("content_hash"))
print("stored status  :", skill.get("status"))
print("experience_ref :", skill.get("experience_ref"))
print("metrics        :", skill.get("metrics"))

if skill.get("content_hash") != INVALID_SKILL_HASH:
    raise RuntimeError(
        "Historical skill identity mismatch."
    )

# ============================================================
# 5. Discover repository/state authority files too
# ============================================================

print("\n[4] AUTHORITY DISCOVERY")
print("-" * 120)

latest_candidates = list(
    INPUT.rglob("DURABILITY-LATEST.json")
)

current_candidates = list(
    INPUT.rglob("CURRENT.json")
)

source_candidates = list(
    INPUT.rglob("SOURCE-OF-TRUTH.json")
)

print("DURABILITY-LATEST :", len(latest_candidates))
print("CURRENT            :", len(current_candidates))
print("SOURCE-OF-TRUTH    :", len(source_candidates))

# ============================================================
# 6. Freeze discovery information for next R3 cell
# ============================================================

WORK = Path("/kaggle/working/RAIOS-V8.6.2/r3")
WORK.mkdir(parents=True, exist_ok=True)

discovery = {
    "skill_file": str(SKILL_FILE),
    "state_root": str(STATE),
    "skill_id": skill.get("skill_id"),
    "content_hash": skill.get("content_hash"),
    "stored_status": skill.get("status"),
    "experience_ref": skill.get("experience_ref"),
    "availability": availability,
    "durability_latest_candidates": [
        str(p) for p in latest_candidates
    ],
    "source_of_truth_candidates": [
        str(p) for p in source_candidates
    ],
}

OUT = WORK / "DYNAMIC-STATE-DISCOVERY.json"

OUT.write_text(
    json.dumps(
        discovery,
        indent=2
    ),
    encoding="utf-8"
)

print("\n" + "=" * 120)
print("R3 DYNAMIC STATE RECOVERY RESULT")
print("=" * 120)

print("Skill file :", SKILL_FILE)
print("State root :", STATE)
print("Output     :", OUT)
print()
print("STATUS: R3_COGNITIVE_STATE_RECOVERED")
print("T4         : KEEP ON")
print("MODEL      : NOT LOADED")
print("TRAINING   : NO")

RAIOS V8.6.2-R3 — DYNAMIC COGNITIVE STATE RECOVERY
T4 PRESERVED | CPU DISCOVERY ONLY | NO MODEL | NO TRAINING

[1] EXACT SKILL DISCOVERY
------------------------------------------------------------------------------------------------------------------------
Exact filename not found.
Searching skill contents...
Content matches: 0

STATUS: HISTORICAL_SKILL_NOT_MOUNTED
The RAIOS cognitive-state dataset is incomplete or not attached
to this T4 session.

ACTION: TURN T4 OFF NOW.


SystemExit: 3

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [1]:
from pathlib import Path
import json

print("=" * 120)
print("RAIOS R3 — PERSISTENT INPUT READINESS GATE")
print("ACCELERATOR NONE | READ ONLY | NO MODEL | NO TRAINING")
print("=" * 120)

INPUT = Path("/kaggle/input")

INVALID_SKILL_HASH = (
    "683ca22e97a2cb919ab078321076bd06"
    "b146b1bf69e937a2062d68755e993b65"
)

# -------------------------------------------------------------------
# 1. Show actual mounted input roots
# -------------------------------------------------------------------

print("\n[1] MOUNTED INPUT ROOTS")
print("-" * 120)

roots = []

if INPUT.exists():
    for p in sorted(INPUT.iterdir()):
        roots.append(p)
        print(" -", p)
else:
    raise RuntimeError("/kaggle/input is missing")

# -------------------------------------------------------------------
# 2. Find RAIOS cognitive state by identity, not fixed path
# -------------------------------------------------------------------

print("\n[2] RAIOS COGNITIVE STATE")
print("-" * 120)

state_candidates = []

for p in INPUT.rglob("DURABILITY-LATEST.json"):
    if p.parent.name == "checkpoints":
        state = p.parent.parent
        state_candidates.append(state)

state_candidates = list(dict.fromkeys(state_candidates))

for state in state_candidates:
    print("STATE CANDIDATE:", state)

if not state_candidates:
    print("RAIOS_STATE = NOT_MOUNTED")
else:
    print("RAIOS_STATE_CANDIDATES =", len(state_candidates))

# -------------------------------------------------------------------
# 3. Exact historical skill
# -------------------------------------------------------------------

print("\n[3] QUARANTINED SKILL")
print("-" * 120)

skill_matches = list(
    INPUT.rglob(f"{INVALID_SKILL_HASH}.json")
)

if skill_matches:
    for p in skill_matches:
        print("FOUND:", p)
else:
    print("NOT FOUND")

# -------------------------------------------------------------------
# 4. Required state areas
# -------------------------------------------------------------------

resolved_state = None

if skill_matches:
    p = skill_matches[0]

    if p.parent.name == "skills":
        resolved_state = p.parent.parent

if resolved_state:
    print("\n[4] REQUIRED COGNITIVE AREAS")
    print("-" * 120)

    required = [
        "skills",
        "experiences",
        "failures",
        "benchmarks",
        "evidence",
        "semantic-corpus",
        "model-profiles",
    ]

    for name in required:
        d = resolved_state / name

        count = (
            len(list(d.glob("*.json")))
            if d.exists()
            else 0
        )

        print(
            f"{name:<20}",
            "YES" if d.exists() else "NO",
            "|",
            count
        )

# -------------------------------------------------------------------
# 5. Persistent model discovery
# -------------------------------------------------------------------

print("\n[5] PERSISTENT MODEL ASSETS")
print("-" * 120)

models = []

for config in INPUT.rglob("config.json"):

    root = config.parent

    low = str(root).lower()

    if any(
        ignored in low
        for ignored in [
            "node_modules",
            ".git",
            ".next",
        ]
    ):
        continue

    tokenizer = any(
        (root / name).exists()
        for name in [
            "tokenizer.json",
            "tokenizer_config.json",
            "tokenizer.model",
            "spiece.model",
        ]
    )

    weights = (
        list(root.glob("*.safetensors"))
        + list(root.glob("pytorch_model*.bin"))
    )

    if tokenizer and weights:
        size = sum(
            p.stat().st_size
            for p in weights
        )

        models.append(
            (root, size)
        )

models = sorted(
    dict(
        (str(p), (p, size))
        for p, size in models
    ).values(),
    key=lambda x: x[1],
    reverse=True,
)

for p, size in models[:20]:
    print(
        p,
        "|",
        round(size / 1024**3, 3),
        "GiB"
    )

if not models:
    print("NO PERSISTENT HF MODEL FOUND")

# -------------------------------------------------------------------
# 6. Final gate
# -------------------------------------------------------------------

state_ready = (
    resolved_state is not None
    and
    (resolved_state / "benchmarks").exists()
    and
    (resolved_state / "evidence").exists()
)

model_ready = len(models) > 0

print("\n" + "=" * 120)
print("R3 PERSISTENT INPUT GATE")
print("=" * 120)

print("Cognitive state :", "READY" if state_ready else "MISSING")
print("Historical skill:", "READY" if skill_matches else "MISSING")
print("Persistent model:", "READY" if model_ready else "MISSING")

if state_ready and model_ready:
    print()
    print("STATUS: R3_PERSISTENT_INPUTS_READY")
    print("NEXT: ENABLE_T4")
else:
    print()
    print("STATUS: R3_PERSISTENT_INPUTS_INCOMPLETE")
    print("NEXT: ATTACH_MISSING_KAGGLE_INPUTS")
    print("DO NOT ENABLE T4")

print("\nGPU USED : NO")
print("TRAINING : NO")

RAIOS R3 — PERSISTENT INPUT READINESS GATE
ACCELERATOR NONE | READ ONLY | NO MODEL | NO TRAINING

[1] MOUNTED INPUT ROOTS
------------------------------------------------------------------------------------------------------------------------

[2] RAIOS COGNITIVE STATE
------------------------------------------------------------------------------------------------------------------------
RAIOS_STATE = NOT_MOUNTED

[3] QUARANTINED SKILL
------------------------------------------------------------------------------------------------------------------------
NOT FOUND

[5] PERSISTENT MODEL ASSETS
------------------------------------------------------------------------------------------------------------------------
NO PERSISTENT HF MODEL FOUND

R3 PERSISTENT INPUT GATE
Cognitive state : MISSING
Historical skill: MISSING
Persistent model: MISSING

STATUS: R3_PERSISTENT_INPUTS_INCOMPLETE
NEXT: ATTACH_MISSING_KAGGLE_INPUTS
DO NOT ENABLE T4

GPU USED : NO
TRAINING : NO


In [1]:
import torch, psutil

print("RAM GB:", round(psutil.virtual_memory().total / 1024**3, 2))
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(i, p.name, round(p.total_memory / 1024**3, 2), "GB VRAM")

RAM GB: 31.35
GPU count: 0


In [2]:
import os, sys, platform, subprocess, shutil, json
from pathlib import Path

def run(cmd):
    try:
        return subprocess.check_output(
            cmd, shell=True, stderr=subprocess.STDOUT, text=True
        ).strip()
    except Exception as e:
        return f"N/A ({e})"

print("=" * 70)
print("RAIOS COMPUTE ENVIRONMENT CENSUS")
print("=" * 70)

# ── OS / Python ──────────────────────────────────────────────
print("\n[ SYSTEM ]")
print("OS:", platform.platform())
print("Python:", sys.version.replace("\n", " "))
print("Architecture:", platform.machine())
print("Processor:", platform.processor() or "N/A")

# ── CPU ──────────────────────────────────────────────────────
print("\n[ CPU ]")
print("Logical CPUs:", os.cpu_count())
print("CPU details:")
print(run("lscpu"))

# ── RAM ──────────────────────────────────────────────────────
print("\n[ MEMORY ]")
try:
    import psutil
    mem = psutil.virtual_memory()
    swap = psutil.swap_memory()

    print(f"RAM total:     {mem.total / 1024**3:.2f} GB")
    print(f"RAM available: {mem.available / 1024**3:.2f} GB")
    print(f"RAM used:      {mem.used / 1024**3:.2f} GB")
    print(f"RAM percent:   {mem.percent}%")
    print(f"Swap total:    {swap.total / 1024**3:.2f} GB")
    print(f"Swap free:     {swap.free / 1024**3:.2f} GB")
except Exception as e:
    print("Memory details unavailable:", e)

# ── NVIDIA GPU / VRAM ────────────────────────────────────────
print("\n[ NVIDIA GPU ]")

if shutil.which("nvidia-smi"):
    print(run(
        "nvidia-smi --query-gpu="
        "index,name,gpu_name,memory.total,memory.free,memory.used,"
        "utilization.gpu,temperature.gpu,driver_version,compute_cap "
        "--format=csv,noheader"
    ))

    print("\nFull NVIDIA-SMI:")
    print(run("nvidia-smi"))
else:
    print("nvidia-smi: NOT FOUND")

# ── PyTorch GPU view ─────────────────────────────────────────
print("\n[ PYTORCH / CUDA ]")
try:
    import torch

    print("PyTorch:", torch.__version__)
    print("CUDA compiled version:", torch.version.cuda)
    print("CUDA available:", torch.cuda.is_available())
    print("CUDA device count:", torch.cuda.device_count())

    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)

        print(f"\nGPU {i}")
        print("Name:", p.name)
        print(f"VRAM: {p.total_memory / 1024**3:.2f} GB")
        print("Compute capability:", f"{p.major}.{p.minor}")

        try:
            free, total = torch.cuda.mem_get_info(i)
            print(f"VRAM free: {free / 1024**3:.2f} GB")
            print(f"VRAM total: {total / 1024**3:.2f} GB")
        except Exception as e:
            print("VRAM runtime info unavailable:", e)

except Exception as e:
    print("PyTorch unavailable/error:", e)

# ── Other accelerator evidence ───────────────────────────────
print("\n[ ACCELERATOR DEVICES ]")
print("/dev/nvidia*:", run("ls -la /dev/nvidia* 2>/dev/null || true"))
print("/dev/dri:", run("ls -la /dev/dri 2>/dev/null || true"))

# ── Storage ──────────────────────────────────────────────────
print("\n[ STORAGE ]")
disk = shutil.disk_usage("/")
print(f"Total: {disk.total / 1024**3:.2f} GB")
print(f"Used:  {disk.used / 1024**3:.2f} GB")
print(f"Free:  {disk.free / 1024**3:.2f} GB")

# ── Relevant ML libraries ────────────────────────────────────
print("\n[ ML STACK ]")
packages = [
    "torch",
    "transformers",
    "accelerate",
    "datasets",
    "sentence_transformers",
    "faiss",
    "faiss_cpu",
    "bitsandbytes",
    "peft",
    "trl",
    "xgboost",
    "sklearn",
]

for pkg in packages:
    try:
        module = __import__(pkg)
        print(f"{pkg}: {getattr(module, '__version__', 'installed')}")
    except Exception:
        print(f"{pkg}: NOT FOUND")

# ── Environment clues ────────────────────────────────────────
print("\n[ ENVIRONMENT ]")
print("Kaggle:", "YES" if os.path.exists("/kaggle") else "NO")
print("Working directory:", os.getcwd())

print("\n" + "=" * 70)
print("CENSUS COMPLETE")
print("=" * 70)

RAIOS COMPUTE ENVIRONMENT CENSUS

[ SYSTEM ]
OS: Linux-6.12.90+-x86_64-with-glibc2.35
Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Architecture: x86_64
Processor: x86_64

[ CPU ]
Logical CPUs: 4
CPU details:
Architecture:                            x86_64
CPU op-mode(s):                          32-bit, 64-bit
Address sizes:                           48 bits physical, 48 bits virtual
Byte Order:                              Little Endian
CPU(s):                                  4
On-line CPU(s) list:                     0-3
Vendor ID:                               AuthenticAMD
Model name:                              AMD EPYC 7B12
CPU family:                              23
Model:                                   49
Thread(s) per core:                      2
Core(s) per socket:                      2
Socket(s):                               1
Stepping:                                0
BogoMIPS:                                4499.99
Flags:                                

In [1]:
import json, os, platform, subprocess, shutil

def cmd(x):
    try:
        p = subprocess.run(
            x,
            shell=True,
            capture_output=True,
            text=True,
            timeout=30
        )
        return {
            "returncode": p.returncode,
            "stdout": p.stdout.strip(),
            "stderr": p.stderr.strip()
        }
    except Exception as e:
        return {"error": repr(e)}

report = {}

report["python"] = platform.python_version()
report["platform"] = platform.platform()

report["memory"] = cmd("free -h")
report["memory_bytes"] = cmd("cat /proc/meminfo | head -20")

report["gpu_smi"] = cmd(
    "nvidia-smi --query-gpu="
    "name,memory.total,memory.free,memory.used,"
    "compute_cap,driver_version "
    "--format=csv,noheader"
)

report["gpu_full"] = cmd("nvidia-smi")

report["disk"] = cmd("df -h /kaggle/working /kaggle/input /tmp 2>/dev/null")

report["cpu"] = cmd("lscpu | head -40")

report["torch"] = {}
try:
    import torch
    report["torch"]["version"] = torch.__version__
    report["torch"]["cuda_available"] = torch.cuda.is_available()
    report["torch"]["cuda_device_count"] = torch.cuda.device_count()

    devices = []
    for i in range(torch.cuda.device_count()):
        prop = torch.cuda.get_device_properties(i)
        devices.append({
            "index": i,
            "name": prop.name,
            "total_memory_bytes": prop.total_memory,
            "major": prop.major,
            "minor": prop.minor,
        })
    report["torch"]["devices"] = devices
except Exception as e:
    report["torch"]["error"] = repr(e)

report["working_bytes_free"] = shutil.disk_usage("/kaggle/working").free
report["working_bytes_total"] = shutil.disk_usage("/kaggle/working").total

print(json.dumps(report, indent=2))

{
  "python": "3.12.13",
  "platform": "Linux-6.12.90+-x86_64-with-glibc2.35",
  "memory": {
    "returncode": 0,
    "stdout": "total        used        free      shared  buff/cache   available\nMem:            31Gi       697Mi        27Gi       1.0Mi       3.0Gi        30Gi\nSwap:             0B          0B          0B",
    "stderr": ""
  },
  "memory_bytes": {
    "returncode": 0,
    "stdout": "MemTotal:       32870488 kB\nMemFree:        28963808 kB\nMemAvailable:   31688836 kB\nBuffers:         1029588 kB\nCached:          1940560 kB\nSwapCached:            0 kB\nActive:          1415692 kB\nInactive:        2049680 kB\nActive(anon):        752 kB\nInactive(anon):   495652 kB\nActive(file):    1414940 kB\nInactive(file):  1554028 kB\nUnevictable:           0 kB\nMlocked:               0 kB\nSwapTotal:             0 kB\nSwapFree:              0 kB\nZswap:                 0 kB\nZswapped:              0 kB\nDirty:              6604 kB\nWriteback:             0 kB",
    "stderr": ""

In [1]:
import json, subprocess, torch, psutil, shutil

def sh(cmd):
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return p.stdout.strip(), p.stderr.strip(), p.returncode

gpu, gpu_err, gpu_rc = sh(
    "nvidia-smi --query-gpu=name,memory.total,memory.free,memory.used "
    "--format=csv,noheader"
)

report = {
    "ram_total_gb": round(psutil.virtual_memory().total / 1024**3, 2),
    "ram_available_gb": round(psutil.virtual_memory().available / 1024**3, 2),
    "cuda_available": torch.cuda.is_available(),
    "cuda_device_count": torch.cuda.device_count(),
    "gpus": [
        {
            "index": i,
            "name": torch.cuda.get_device_name(i),
            "vram_gb": round(
                torch.cuda.get_device_properties(i).total_memory / 1024**3, 2
            ),
        }
        for i in range(torch.cuda.device_count())
    ],
    "nvidia_smi": gpu,
    "working_free_gb": round(shutil.disk_usage("/kaggle/working").free / 1024**3, 2),
    "working_total_gb": round(shutil.disk_usage("/kaggle/working").total / 1024**3, 2),
}

print(json.dumps(report, indent=2))

{
  "ram_total_gb": 31.35,
  "ram_available_gb": 29.74,
  "cuda_available": true,
  "cuda_device_count": 2,
  "gpus": [
    {
      "index": 0,
      "name": "Tesla T4",
      "vram_gb": 14.56
    },
    {
      "index": 1,
      "name": "Tesla T4",
      "vram_gb": 14.56
    }
  ],
  "nvidia_smi": "Tesla T4, 15360 MiB, 14912 MiB, 0 MiB\nTesla T4, 15360 MiB, 14912 MiB, 0 MiB",
  "working_free_gb": 19.5,
  "working_total_gb": 19.52
}


In [1]:

import json, os, platform, shutil, subprocess, torch, psutil, hashlib, time

WORKER_ID = "KAGGLE_A2"   # في الحساب الثاني غيّرها إلى KAGGLE_B

def cmd(command):
    p = subprocess.run(
        command,
        shell=True,
        capture_output=True,
        text=True
    )
    return {
        "rc": p.returncode,
        "stdout": p.stdout.strip(),
        "stderr": p.stderr.strip()
    }

gpus = []

for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    gpus.append({
        "index": i,
        "name": p.name,
        "vram_bytes": p.total_memory,
        "vram_gb": round(p.total_memory / 1024**3, 2),
        "compute_capability": f"{p.major}.{p.minor}"
    })

nonce = hashlib.sha256(
    f"{WORKER_ID}-{time.time_ns()}".encode()
).hexdigest()

report = {
    "worker_id": WORKER_ID,
    "runtime": {
        "python": platform.python_version(),
        "torch": torch.__version__,
        "cuda_available": torch.cuda.is_available(),
        "cuda_device_count": torch.cuda.device_count()
    },
    "hardware": {
        "cpu_count": os.cpu_count(),
        "ram_total_gb": round(psutil.virtual_memory().total / 1024**3, 2),
        "ram_available_gb": round(psutil.virtual_memory().available / 1024**3, 2),
        "gpus": gpus,
        "nvidia_smi": cmd(
            "nvidia-smi --query-gpu=name,memory.total,memory.free,memory.used "
            "--format=csv,noheader"
        )
    },
    "storage": {
        "working_total_gb": round(shutil.disk_usage("/kaggle/working").total / 1024**3, 2),
        "working_free_gb": round(shutil.disk_usage("/kaggle/working").free / 1024**3, 2)
    },
    "nonce": nonce,
    "hardware_state": (
        "GPU_AVAILABLE"
        if torch.cuda.is_available()
        else "CPU_ONLY"
    )
}

path = f"/kaggle/working/{WORKER_ID}-HARDWARE-PROOF.json"

with open(path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)

with open(path, "rb") as f:
    digest = hashlib.sha256(f.read()).hexdigest()

print(json.dumps({
    "worker_id": WORKER_ID,
    "hardware_state": report["hardware_state"],
    "ram_total_gb": report["hardware"]["ram_total_gb"],
    "ram_available_gb": report["hardware"]["ram_available_gb"],
    "gpu_count": len(gpus),
    "gpus": gpus,
    "working_free_gb": report["storage"]["working_free_gb"],
    "proof_file": path,
    "proof_sha256": digest
}, indent=2))A2

NameError: name 'KAGGLE_A2' is not defined

In [1]:
WORKER_ID = "KAGGLE_A2"

print("WORKER_ID =", WORKER_ID)
print("PYTHON_CELL_OK = TRUE")

WORKER_ID = KAGGLE_A2
PYTHON_CELL_OK = TRUE


In [2]:
import json
import os
import platform
import shutil
import subprocess

WORKER_ID = "KAGGLE_A2"

try:
    import torch
except Exception:
    torch = None

try:
    import psutil
except Exception:
    psutil = None


def run(command):
    try:
        p = subprocess.run(
            command,
            shell=True,
            capture_output=True,
            text=True,
            timeout=30,
        )
        return {
            "returncode": p.returncode,
            "stdout": p.stdout.strip(),
            "stderr": p.stderr.strip(),
        }
    except Exception as e:
        return {
            "returncode": -1,
            "stdout": "",
            "stderr": repr(e),
        }


ram_total_gb = None
ram_available_gb = None

if psutil is not None:
    memory = psutil.virtual_memory()
    ram_total_gb = round(memory.total / 1024**3, 2)
    ram_available_gb = round(memory.available / 1024**3, 2)


cuda_available = False
gpu_count = 0
gpus = []

if torch is not None:
    cuda_available = bool(torch.cuda.is_available())
    gpu_count = int(torch.cuda.device_count())

    for i in range(gpu_count):
        p = torch.cuda.get_device_properties(i)

        gpus.append({
            "index": i,
            "name": p.name,
            "vram_gb": round(
                p.total_memory / 1024**3,
                2
            ),
        })


working = shutil.disk_usage("/kaggle/working")

report = {
    "worker_id": WORKER_ID,

    "runtime": {
        "hostname": platform.node(),
        "python": platform.python_version(),
    },

    "hardware": {
        "cpu_count": os.cpu_count(),
        "ram_total_gb": ram_total_gb,
        "ram_available_gb": ram_available_gb,
        "cuda_available": cuda_available,
        "gpu_count": gpu_count,
        "gpus": gpus,
        "nvidia_smi": run(
            "nvidia-smi --query-gpu="
            "name,memory.total,memory.free,memory.used "
            "--format=csv,noheader"
        ),
    },

    "storage": {
        "working_total_gb": round(
            working.total / 1024**3,
            2
        ),
        "working_free_gb": round(
            working.free / 1024**3,
            2
        ),
    },
}

print(json.dumps(report, indent=2))

{
  "worker_id": "KAGGLE_A2",
  "runtime": {
    "hostname": "936472ed77c1",
    "python": "3.12.13"
  },
  "hardware": {
    "cpu_count": 4,
    "ram_total_gb": 31.35,
    "ram_available_gb": 29.7,
    "cuda_available": true,
    "gpu_count": 2,
    "gpus": [
      {
        "index": 0,
        "name": "Tesla T4",
        "vram_gb": 14.56
      },
      {
        "index": 1,
        "name": "Tesla T4",
        "vram_gb": 14.56
      }
    ],
    "nvidia_smi": {
      "returncode": 0,
      "stdout": "Tesla T4, 15360 MiB, 14909 MiB, 3 MiB\nTesla T4, 15360 MiB, 14909 MiB, 3 MiB",
      "stderr": ""
    }
  },
  "storage": {
    "working_total_gb": 19.52,
    "working_free_gb": 19.5
  }
}


In [1]:
NOTEBOOK_SLUG = "greenylife/notebook8d96bf4e18"
# ============================================================
# RAIOS — KAGGLE NOTEBOOK DEEP SELF-IDENTITY + GIT BINDING
# TASK_ID=RAIOS-KAGGLE-NOTEBOOK-DEEP-CENSUS-01
#
# PURPOSE:
#   Identify exactly what this notebook is, what is mounted,
#   what exists in /kaggle/working, and whether a REAL Git
#   repository / GitHub relationship exists inside this runtime.
#
# SAFE:
#   - NO GPU/TPU
#   - NO model execution
#   - NO install
#   - NO fetch/pull/push
#   - NO delete
#   - NO secret values
#   - NO Kaggle API calls
#
# WRITES REPORTS ONLY UNDER /kaggle/working/
# ============================================================

from __future__ import annotations

import os
import re
import sys
import json
import hashlib
import platform
import subprocess
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter, defaultdict

# ------------------------------------------------------------
# REQUIRED NOTEBOOK ID
# ------------------------------------------------------------

try:
    NOTEBOOK_SLUG
except NameError:
    raise RuntimeError(
        "Set NOTEBOOK_SLUG first, e.g. "
        "'greenylife/notebook8c2d6a9080'"
    )

TASK_ID = "RAIOS-KAGGLE-NOTEBOOK-DEEP-CENSUS-01"
UTC_NOW = datetime.now(timezone.utc).isoformat()

ROOT = Path("/kaggle")
INPUT = Path("/kaggle/input")
WORKING = Path("/kaggle/working")

SAFE_SLUG = NOTEBOOK_SLUG.replace("/", "__")

OUT_JSON = WORKING / f"RAIOS-KAGGLE-CENSUS__{SAFE_SLUG}.json"
OUT_MD = WORKING / f"RAIOS-KAGGLE-CENSUS__{SAFE_SLUG}.md"
OUT_SHA = WORKING / f"RAIOS-KAGGLE-CENSUS__{SAFE_SLUG}.sha256"

# ------------------------------------------------------------
# SAFETY / LIMITS
# ------------------------------------------------------------

SKIP_DIRS = {
    ".venv", "venv", "node_modules", ".next", "__pycache__",
    ".cache", ".npm", ".yarn", "dist", "build", "coverage",
    ".turbo",
}

SECRET_PATTERNS = (
    "token", "secret", "password", "passwd", "credential",
    "private_key", "id_rsa", ".pem", ".key", ".env",
    "kaggle.json", "access_token",
)

TEXT_EXTENSIONS = {
    ".py", ".ps1", ".sh", ".md", ".txt", ".json", ".jsonl",
    ".yaml", ".yml", ".toml", ".ini", ".cfg",
    ".ts", ".tsx", ".js", ".jsx", ".mjs", ".cjs",
    ".sql", ".csv", ".ipynb",
}

MAX_FILES = 25000
MAX_TEXT_FILES = 1500
MAX_TEXT_READ = 64 * 1024
MAX_HASH_SIZE = 32 * 1024 * 1024

# ------------------------------------------------------------
# PROJECT SIGNALS
# ------------------------------------------------------------

SIGNALS = {
    "RAIOS": [
        "raios", ".ai-os", "ai-os"
    ],
    "GREENY_LIFE": [
        "greeny-life", "greeny life", "greenylife"
    ],
    "RAIOS_V8": [
        "raios v8", "raios/v8"
    ],
    "RAIOS_V9": [
        "raios v9", "raios/v9"
    ],
    "RIF": [
        "research intelligence factory", "rif"
    ],
    "LIVE_BRAIN": [
        "live brain", "live_brain"
    ],
    "EVOLUTION_BRAIN": [
        "evolution brain", "evolution_brain"
    ],
    "AUTONOMIC_CORE": [
        "autonomic core", "autonomic_core", "autonomic"
    ],
    "ENTERPRISE_BRAIN": [
        "enterprise brain", "enterprise_brain"
    ],
    "MAIN_CORTEX": [
        "main cortex", "main_cortex"
    ],
    "COGNITIVE_WAL": [
        "cognitive wal", "cognitive_wal"
    ],
    "LEARNING": [
        "learning",
        "experience_event",
        "skill compilation",
        "skill_compilation",
        "discovered",
        "validated",
        "canonical",
    ],
    "MODEL_FACTORY": [
        "model factory", "model_factory", "distillation"
    ],
    "MODEL_ECOLOGY": [
        "model ecology", "model_ecology"
    ],
    "NEUROLINGUA": [
        "neurolingua", "neuro lingua"
    ],
    "QWEN": [
        "qwen", "qwen3"
    ],
    "GRANITE": [
        "granite", "ibm granite"
    ],
    "DEEPSEEK": [
        "deepseek"
    ],
    "GLM": [
        "glm"
    ],
    "LION": [
        "lion"
    ],
    "COMMAND_FABRIC": [
        "command fabric", "command_fabric", "cicf"
    ],
    "MCP": [
        "model context protocol", "mcp"
    ],
    "NATS": [
        "nats", "jetstream"
    ],
    "GL002": [
        "gl-002", "gl002", "main-brain"
    ],
    "GL003": [
        "gl-003", "gl003", "project-brains"
    ],
    "GL004": [
        "gl-004", "gl004"
    ],
    "GL005": [
        "gl-005", "gl005"
    ],
    "GELS": [
        "gels", "global export label standard"
    ],
    "GL_DOS": [
        "gl-dos", "gldos"
    ],
    "EOS": [
        "enterprise operating system", "eos"
    ],
}

# ------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------

def sha256_file(path: Path):
    h = hashlib.sha256()
    try:
        with path.open("rb") as f:
            for block in iter(lambda: f.read(1024 * 1024), b""):
                h.update(block)
        return h.hexdigest()
    except Exception:
        return None


def safe_cmd(args, cwd=None, timeout=15):
    try:
        p = subprocess.run(
            args,
            cwd=str(cwd) if cwd else None,
            capture_output=True,
            text=True,
            encoding="utf-8",
            errors="replace",
            timeout=timeout,
        )
        return {
            "ok": p.returncode == 0,
            "rc": p.returncode,
            "stdout": (p.stdout or "").strip(),
            "stderr": (p.stderr or "").strip(),
        }
    except Exception as exc:
        return {
            "ok": False,
            "rc": None,
            "stdout": "",
            "stderr": f"{type(exc).__name__}: {exc}",
        }


def secret_like(path: Path):
    low = str(path).lower()
    return any(x in low for x in SECRET_PATTERNS)


def redact_url(value):
    if not value:
        return value

    # https://TOKEN@github.com/...
    value = re.sub(
        r"(https?://)[^/@\s]+@",
        r"\1<REDACTED>@",
        value,
        flags=re.I,
    )

    # x-access-token:TOKEN
    value = re.sub(
        r"x-access-token:[^@\s]+",
        "x-access-token:<REDACTED>",
        value,
        flags=re.I,
    )

    return value


def human_size(size):
    if size is None:
        return None

    value = float(size)

    for unit in ("B", "KB", "MB", "GB", "TB"):
        if value < 1024 or unit == "TB":
            return f"{value:.2f} {unit}"
        value /= 1024


def walk_files(root, limit=MAX_FILES):
    if not root.exists():
        return

    emitted = 0

    for base, dirs, files in os.walk(root):
        dirs[:] = [d for d in dirs if d not in SKIP_DIRS]

        for name in files:
            if emitted >= limit:
                return

            emitted += 1
            yield Path(base) / name


# ------------------------------------------------------------
# DATASET MOUNTS
# ------------------------------------------------------------

datasets = []

if INPUT.exists():
    for dataset_dir in sorted(INPUT.iterdir()):
        if not dataset_dir.is_dir():
            continue

        total_files = 0
        total_size = 0
        extensions = Counter()
        examples = []
        important = []

        for p in walk_files(dataset_dir):
            total_files += 1

            try:
                size = p.stat().st_size
            except Exception:
                size = 0

            total_size += size
            extensions[p.suffix.lower() or "<none>"] += 1

            if len(examples) < 30:
                examples.append({
                    "path": str(p),
                    "size": size,
                })

            low = p.name.lower()

            if any(
                x in low
                for x in (
                    "raios", "greeny", "manifest", "state",
                    "evidence", "version", "brain", "model",
                    "checkpoint", "knowledge", "skill",
                    "architecture", "decision",
                )
            ):
                if len(important) < 100:
                    important.append({
                        "path": str(p),
                        "size": size,
                        "sha256": (
                            sha256_file(p)
                            if size <= MAX_HASH_SIZE and not secret_like(p)
                            else None
                        ),
                    })

        datasets.append({
            "mount_name": dataset_dir.name,
            "mount_path": str(dataset_dir),
            "files": total_files,
            "bytes": total_size,
            "human_size": human_size(total_size),
            "extensions": dict(extensions.most_common(40)),
            "sample_files": examples,
            "important_files": important,
        })


# ------------------------------------------------------------
# WORKING DIRECTORY
# ------------------------------------------------------------

working_files = []
working_total_size = 0
working_extensions = Counter()

if WORKING.exists():
    for p in walk_files(WORKING):
        # Exclude reports this cell creates
        if p in {OUT_JSON, OUT_MD, OUT_SHA}:
            continue

        try:
            size = p.stat().st_size
        except Exception:
            size = None

        if size:
            working_total_size += size

        working_extensions[p.suffix.lower() or "<none>"] += 1

        if len(working_files) < 500:
            working_files.append({
                "path": str(p),
                "size": size,
                "sha256": (
                    sha256_file(p)
                    if (
                        size is not None
                        and size <= MAX_HASH_SIZE
                        and not secret_like(p)
                    )
                    else None
                ),
                "secret_like_name": secret_like(p),
                "secret_content_read": False,
            })


# ------------------------------------------------------------
# STATIC PROJECT SIGNALS
# ------------------------------------------------------------

signal_hits = defaultdict(list)
sampled_text_files = 0

for scan_root in (WORKING, INPUT):
    if not scan_root.exists():
        continue

    for p in walk_files(scan_root):
        if sampled_text_files >= MAX_TEXT_FILES:
            break

        if p.suffix.lower() not in TEXT_EXTENSIONS:
            continue

        if secret_like(p):
            continue

        try:
            if p.stat().st_size > 4 * 1024 * 1024:
                continue

            text = p.read_text(
                encoding="utf-8",
                errors="ignore"
            )[:MAX_TEXT_READ]

            sampled_text_files += 1

            combined = (
                str(p).lower()
                + "\n"
                + text.lower()
            )

            for family, needles in SIGNALS.items():
                if any(n.lower() in combined for n in needles):
                    if len(signal_hits[family]) < 25:
                        signal_hits[family].append(str(p))

        except Exception:
            continue

signal_summary = sorted(
    [
        {
            "family": family,
            "matched_files": len(paths),
            "evidence_paths": paths[:12],
        }
        for family, paths in signal_hits.items()
    ],
    key=lambda x: (-x["matched_files"], x["family"])
)


# ------------------------------------------------------------
# REAL GIT REPOSITORY DISCOVERY
# ------------------------------------------------------------

git_roots = set()

for base_root in (WORKING, INPUT):
    if not base_root.exists():
        continue

    for base, dirs, files in os.walk(base_root):
        p = Path(base)

        # A real .git directory or worktree .git file.
        if ".git" in dirs or ".git" in files:
            git_roots.add(str(p))
            dirs[:] = []
            continue

        dirs[:] = [
            d for d in dirs
            if d not in SKIP_DIRS
        ]


git_repositories = []

for repo_string in sorted(git_roots):
    repo = Path(repo_string)

    inside = safe_cmd(
        ["git", "rev-parse", "--is-inside-work-tree"],
        cwd=repo,
    )

    if not (
        inside["ok"]
        and inside["stdout"].lower() == "true"
    ):
        continue

    head = safe_cmd(
        ["git", "rev-parse", "HEAD"],
        cwd=repo,
    )

    branch = safe_cmd(
        ["git", "branch", "--show-current"],
        cwd=repo,
    )

    status = safe_cmd(
        ["git", "status", "--porcelain=v1"],
        cwd=repo,
    )

    shallow = safe_cmd(
        ["git", "rev-parse", "--is-shallow-repository"],
        cwd=repo,
    )

    remotes = safe_cmd(
        ["git", "remote", "-v"],
        cwd=repo,
    )

    config_origin = safe_cmd(
        ["git", "config", "--get", "remote.origin.url"],
        cwd=repo,
    )

    recent = safe_cmd(
        [
            "git", "log", "-10",
            "--format=%H%x09%P%x09%aI%x09%s"
        ],
        cwd=repo,
    )

    remote_lines = []

    for line in remotes["stdout"].splitlines():
        remote_lines.append(redact_url(line))

    github_remote_detected = any(
        "github.com" in line.lower()
        for line in remote_lines
    )

    git_repositories.append({
        "repo_root": str(repo),
        "head": head["stdout"] if head["ok"] else None,
        "branch": branch["stdout"] if branch["ok"] else None,
        "shallow": (
            shallow["stdout"]
            if shallow["ok"]
            else None
        ),
        "dirty_entries": len([
            x for x in status["stdout"].splitlines()
            if x.strip()
        ]),
        "origin_redacted": (
            redact_url(config_origin["stdout"])
            if config_origin["ok"]
            else None
        ),
        "remotes_redacted": remote_lines,
        "github_remote_detected": github_remote_detected,
        "recent_commits": recent["stdout"].splitlines(),
        "fetch_executed": False,
        "pull_executed": False,
        "push_executed": False,
        "remote_state_contacted": False,
    })


# ------------------------------------------------------------
# DETERMINE GITHUB RELATIONSHIP TYPE
# ------------------------------------------------------------

if git_repositories:
    github_repos = [
        r for r in git_repositories
        if r["github_remote_detected"]
    ]

    if github_repos:
        git_relation = (
            "REAL_LOCAL_GIT_REPOSITORY_WITH_GITHUB_REMOTE"
        )
    else:
        git_relation = (
            "REAL_LOCAL_GIT_REPOSITORY_WITHOUT_GITHUB_REMOTE"
        )
else:
    git_relation = "NO_LOCAL_GIT_REPOSITORY_DETECTED"


# ------------------------------------------------------------
# KAGGLE RUNTIME METADATA — SAFE ENV ONLY
# ------------------------------------------------------------

safe_env_names = [
    "KAGGLE_KERNEL_RUN_TYPE",
    "KAGGLE_URL_BASE",
    "KAGGLE_DOCKER_IMAGE",
    "KAGGLE_DATA_PROXY_PROJECT",
    "CUDA_VISIBLE_DEVICES",
    "NVIDIA_VISIBLE_DEVICES",
]

safe_environment = {}

for key in safe_env_names:
    if key in os.environ:
        safe_environment[key] = os.environ[key]


# ------------------------------------------------------------
# SECRET PRESENCE — NAMES ONLY
# ------------------------------------------------------------

secret_like_paths = []

for root in (WORKING, INPUT):
    if not root.exists():
        continue

    for p in walk_files(root):
        if secret_like(p):
            if len(secret_like_paths) < 100:
                secret_like_paths.append(str(p))


# ------------------------------------------------------------
# CLASSIFICATION
# ------------------------------------------------------------

ranked_families = [
    x["family"]
    for x in signal_summary[:12]
]

if "RAIOS_V9" in ranked_families:
    project_classification = "RAIOS_V9_RELATED"
elif "RAIOS_V8" in ranked_families:
    project_classification = "RAIOS_V8_RELATED"
elif "RAIOS" in ranked_families and "NEUROLINGUA" in ranked_families:
    project_classification = "RAIOS_NEUROLINGUA_RELATED"
elif "RAIOS" in ranked_families:
    project_classification = "RAIOS_RELATED"
elif "GREENY_LIFE" in ranked_families:
    project_classification = "GREENY_LIFE_RELATED"
else:
    project_classification = "PURPOSE_UNRESOLVED"


# ------------------------------------------------------------
# REPORT
# ------------------------------------------------------------

report = {
    "schema": "raios.kaggle.notebook.deep-self-census.v1",
    "task_id": TASK_ID,
    "generated_at_utc": UTC_NOW,

    "notebook": {
        "slug": NOTEBOOK_SLUG,
        "owner": NOTEBOOK_SLUG.split("/", 1)[0],
        "notebook_id": NOTEBOOK_SLUG.split("/", 1)[1],
        "identity_source": "C1_MANUAL_BINDING_IN_CELL",
    },

    "evidence_class": "LIVE_KAGGLE_RUNTIME_LOCAL_INSPECTION",

    "safety": {
        "gpu_started": False,
        "tpu_started": False,
        "model_run": False,
        "training_run": False,
        "inference_run": False,
        "kaggle_api_called": False,
        "network_call_intentionally_made": False,
        "git_fetch": False,
        "git_pull": False,
        "git_push": False,
        "delete": False,
        "secret_contents_read": False,
        "only_report_outputs_written": True,
    },

    "runtime": {
        "hostname": platform.node(),
        "platform": platform.platform(),
        "python": sys.version.split()[0],
        "cwd": os.getcwd(),
        "kaggle_root_exists": ROOT.exists(),
        "input_exists": INPUT.exists(),
        "working_exists": WORKING.exists(),
        "safe_environment": safe_environment,
    },

    "classification": {
        "deterministic_classification": project_classification,
        "runtime_capability_proof": False,
        "signals": signal_summary,
    },

    "datasets": {
        "mounted_count": len(datasets),
        "items": datasets,
    },

    "working": {
        "files_observed": len(working_files),
        "bytes_observed": working_total_size,
        "human_size": human_size(working_total_size),
        "extensions": dict(
            working_extensions.most_common(50)
        ),
        "files": working_files,
    },

    "git": {
        "relationship_classification": git_relation,
        "repositories_found": len(git_repositories),
        "repositories": git_repositories,

        "github_push_proven": False,

        "important_law": (
            "A GitHub remote or local commit does not prove "
            "that Kaggle pushed the commit to GitHub."
        ),
    },

    "secret_safety": {
        "secret_like_paths": secret_like_paths,
        "contents_read": False,
        "contents_reported": False,
    },

    "proof_limits": {
        "file_presence_ne_runtime": True,
        "git_remote_ne_push": True,
        "git_commit_ne_remote_commit": True,
        "mounted_dataset_ne_unique_value": True,
        "model_asset_ne_assimilation": True,
        "classification_ne_capability_proof": True,
    },
}

WORKING.mkdir(parents=True, exist_ok=True)

OUT_JSON.write_text(
    json.dumps(
        report,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

json_sha = sha256_file(OUT_JSON)

# ------------------------------------------------------------
# HUMAN REPORT
# ------------------------------------------------------------

lines = [
    "# RAIOS Kaggle Notebook Deep Census",
    "",
    f"- Notebook: `{NOTEBOOK_SLUG}`",
    f"- Generated UTC: `{UTC_NOW}`",
    f"- Evidence: `LIVE_KAGGLE_RUNTIME_LOCAL_INSPECTION`",
    f"- Classification: `{project_classification}`",
    f"- Mounted datasets: `{len(datasets)}`",
    f"- Git repositories: `{len(git_repositories)}`",
    f"- Git relationship: `{git_relation}`",
    f"- JSON SHA256: `{json_sha}`",
    "",
    "## Signals",
]

for row in signal_summary[:20]:
    lines.append(
        f"- {row['family']}: {row['matched_files']} files"
    )

lines.extend([
    "",
    "## Mounted datasets",
])

for row in datasets:
    lines.append(
        f"- `{row['mount_name']}`: "
        f"{row['files']} files / {row['human_size']}"
    )

lines.extend([
    "",
    "## Git repositories",
])

if not git_repositories:
    lines.append("- None detected.")
else:
    for repo in git_repositories:
        lines.append(
            f"- `{repo['repo_root']}` | "
            f"HEAD=`{repo['head']}` | "
            f"branch=`{repo['branch']}` | "
            f"GitHub={repo['github_remote_detected']}"
        )

OUT_MD.write_text(
    "\n".join(lines),
    encoding="utf-8",
)

md_sha = sha256_file(OUT_MD)

OUT_SHA.write_text(
    f"{json_sha}  {OUT_JSON.name}\n"
    f"{md_sha}  {OUT_MD.name}\n",
    encoding="utf-8",
)

# ------------------------------------------------------------
# C1 / C2 / C3 / C6 COMPACT RETURN
# ------------------------------------------------------------

print()
print("=" * 80)
print("RAIOS KAGGLE NOTEBOOK DEEP CENSUS")
print("=" * 80)

print(f"NOTEBOOK={NOTEBOOK_SLUG}")
print("EVIDENCE_CLASS=LIVE_KAGGLE_RUNTIME_LOCAL_INSPECTION")

print(
    f"DETERMINISTIC_CLASSIFICATION="
    f"{project_classification}"
)

print(
    f"MOUNTED_DATASETS={len(datasets)}"
)

for d in datasets:
    print(
        f"DATASET={d['mount_name']} | "
        f"FILES={d['files']} | "
        f"SIZE={d['human_size']}"
    )

print(
    f"GIT_REPOS_FOUND={len(git_repositories)}"
)

print(
    f"GIT_RELATIONSHIP={git_relation}"
)

for r in git_repositories:
    print(
        f"GIT_REPO={r['repo_root']}"
    )
    print(
        f"GIT_HEAD={r['head']}"
    )
    print(
        f"GIT_BRANCH={r['branch']}"
    )
    print(
        f"GITHUB_REMOTE_DETECTED="
        f"{r['github_remote_detected']}"
    )
    print(
        f"GIT_ORIGIN={r['origin_redacted']}"
    )

print("GITHUB_PUSH_PROVEN=false")
print("MODEL_EXECUTION=false")
print("GPU_TPU_STARTED=false")
print("NETWORK_CALLS_INTENTIONALLY_MADE=false")
print("SECRET_CONTENT_READ=false")

print()
print("TOP_SIGNALS:")

for row in signal_summary[:15]:
    print(
        f"  {row['family']}="
        f"{row['matched_files']}"
    )

print()
print(f"REPORT_JSON={OUT_JSON}")
print(f"REPORT_JSON_SHA256={json_sha}")

print(f"REPORT_MD={OUT_MD}")
print(f"REPORT_MD_SHA256={md_sha}")

print(f"HASH_SIDECAR={OUT_SHA}")

print()
print(
    "FINAL_VERDICT="
    "KAGGLE_NOTEBOOK_LIVE_IDENTITY_READY_FOR_RECONCILIATION"
)

print("=" * 80)


RAIOS KAGGLE NOTEBOOK DEEP CENSUS
NOTEBOOK=greenylife/notebook8d96bf4e18
EVIDENCE_CLASS=LIVE_KAGGLE_RUNTIME_LOCAL_INSPECTION
DETERMINISTIC_CLASSIFICATION=RAIOS_V8_RELATED
MOUNTED_DATASETS=1
DATASET=datasets | FILES=124 | SIZE=341.39 KB
GIT_REPOS_FOUND=0
GIT_RELATIONSHIP=NO_LOCAL_GIT_REPOSITORY_DETECTED
GITHUB_PUSH_PROVEN=false
MODEL_EXECUTION=false
GPU_TPU_STARTED=false
NETWORK_CALLS_INTENTIONALLY_MADE=false
SECRET_CONTENT_READ=false

TOP_SIGNALS:
  GREENY_LIFE=25
  LEARNING=25
  RAIOS=25
  RIF=25
  QWEN=7
  GELS=4
  GL005=2
  RAIOS_V8=2
  EOS=1
  GL_DOS=1
  MODEL_FACTORY=1

REPORT_JSON=/kaggle/working/RAIOS-KAGGLE-CENSUS__greenylife__notebook8d96bf4e18.json
REPORT_JSON_SHA256=d67278a6462617756fc4cc936e3adfc074714d8de271e89625719e2c0e47183c
REPORT_MD=/kaggle/working/RAIOS-KAGGLE-CENSUS__greenylife__notebook8d96bf4e18.md
REPORT_MD_SHA256=b7c4c856ba97125b1371050fa6857a6d53391d3d926618c832828f63190560b5
HASH_SIDECAR=/kaggle/working/RAIOS-KAGGLE-CENSUS__greenylife__notebook8d96bf4e18.sha2

In [2]:
NOTEBOOK_SLUG = "greenylife/notebook8d96bf4e18"
# للثاني:
# NOTEBOOK_SLUG = "greenylife/notebook8d96bf4e18"

# ============================================================
# RAIOS — KAGGLE DEEP ASSET / LINEAGE MANIFEST
# TASK_ID=RAIOS-KAGGLE-ASSET-LINEAGE-01
#
# NO NETWORK
# NO GPU/TPU
# NO MODEL
# NO INSTALL
# NO GIT FETCH/PULL/PUSH
# NO DELETE
# NO SECRET CONTENT
# ============================================================

from __future__ import annotations

import os
import re
import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter, defaultdict

TASK_ID = "RAIOS-KAGGLE-ASSET-LINEAGE-01"
NOW = datetime.now(timezone.utc).isoformat()

INPUT = Path("/kaggle/input")
WORKING = Path("/kaggle/working")

SAFE_SLUG = NOTEBOOK_SLUG.replace("/", "__")

OUT = WORKING / f"RAIOS-KAGGLE-ASSET-LINEAGE__{SAFE_SLUG}.json"
OUT_SHA = WORKING / f"RAIOS-KAGGLE-ASSET-LINEAGE__{SAFE_SLUG}.sha256"

SECRET_PATTERNS = (
    "token", "secret", "password", "passwd", "credential",
    "private_key", "id_rsa", ".pem", ".key", ".env",
    "kaggle.json", "access_token"
)

TEXT_EXTS = {
    ".json", ".jsonl", ".md", ".txt", ".py",
    ".ps1", ".sh", ".yaml", ".yml",
    ".toml", ".ini", ".cfg",
    ".ts", ".tsx", ".js", ".jsx",
    ".ipynb"
}

# Important lineage/version indicators.
SEARCH_TERMS = [
    "RAIOS",
    "RAIOS V8",
    "RAIOS_V8",
    "RAIOS/V8",
    "RAIOS V9",
    "RAIOS_V9",
    "RAIOS/V9",

    "Live Brain",
    "Evolution Brain",
    "Autonomic Core",
    "Enterprise Brain",
    "Main Cortex",

    "RIF",
    "Model Factory",
    "Model Ecology",

    "Cognitive WAL",
    "DISCOVERED",
    "VALIDATED",
    "CANONICAL",

    "NeuroLingua",

    "GL-002",
    "GL002",
    "GL-003",
    "GL003",
    "GL-004",
    "GL004",
    "GL-005",
    "GL005",

    "Qwen",
    "Granite",
    "DeepSeek",
    "Lion",
    "GLM",

    "MCP",
    "NATS",
    "JetStream",
    "Command Fabric",
    "CICF",

    "Greeny-Life",
    "GL-DOS",
    "GELS",
    "EOS",

    "12603d02253547c7727bc84ce68c318e8e9258bc",
    "da67f449",
    "aace3d8",
    "c336fc88",
    "30637e7821c0c58d16e2ef57e902428a14396be1",
]


def secret_like(path: Path) -> bool:
    low = str(path).lower()
    return any(x in low for x in SECRET_PATTERNS)


def sha256_file(path: Path):
    try:
        h = hashlib.sha256()
        with path.open("rb") as f:
            for chunk in iter(lambda: f.read(1024 * 1024), b""):
                h.update(chunk)
        return h.hexdigest()
    except Exception:
        return None


def safe_rel(path: Path, root: Path):
    try:
        return str(path.relative_to(root)).replace("\\", "/")
    except Exception:
        return str(path)


def all_files():
    rows = []

    for surface_name, root in (
        ("INPUT", INPUT),
        ("WORKING", WORKING),
    ):
        if not root.exists():
            continue

        for base, dirs, files in os.walk(root):
            dirs[:] = [
                d for d in dirs
                if d not in {
                    ".git", "__pycache__", ".cache",
                    ".venv", "venv", "node_modules",
                    ".next"
                }
            ]

            for name in files:
                p = Path(base) / name

                # Don't census the output we are currently generating.
                if p in {OUT, OUT_SHA}:
                    continue

                try:
                    size = p.stat().st_size
                except Exception:
                    size = None

                rows.append({
                    "surface": surface_name,
                    "absolute_path": str(p),
                    "relative_path": safe_rel(p, root),
                    "name": p.name,
                    "extension": p.suffix.lower(),
                    "size": size,
                    "sha256": (
                        None if secret_like(p)
                        else sha256_file(p)
                    ),
                    "secret_like": secret_like(p),
                    "secret_content_read": False,
                })

    return rows


files = all_files()

# ------------------------------------------------------------
# DIRECTORY SUMMARY
# ------------------------------------------------------------

directory_counts = Counter()
extension_counts = Counter()

for row in files:
    p = Path(row["absolute_path"])

    try:
        parent = str(p.parent)
    except Exception:
        parent = ""

    directory_counts[parent] += 1
    extension_counts[row["extension"] or "<none>"] += 1


# ------------------------------------------------------------
# EXACT DUPLICATES INSIDE NOTEBOOK ENVIRONMENT
# ------------------------------------------------------------

hash_groups = defaultdict(list)

for row in files:
    digest = row.get("sha256")
    if digest:
        hash_groups[digest].append(row["absolute_path"])

duplicate_groups = [
    {
        "sha256": digest,
        "count": len(paths),
        "paths": paths,
    }
    for digest, paths in hash_groups.items()
    if len(paths) > 1
]


# ------------------------------------------------------------
# STATIC TEXT / LINEAGE SEARCH
# ------------------------------------------------------------

term_hits = defaultdict(list)

MAX_TEXT_BYTES = 512 * 1024

for row in files:
    p = Path(row["absolute_path"])

    if row["secret_like"]:
        continue

    if p.suffix.lower() not in TEXT_EXTS:
        continue

    try:
        if p.stat().st_size > 8 * 1024 * 1024:
            continue

        text = p.read_text(
            encoding="utf-8",
            errors="ignore"
        )[:MAX_TEXT_BYTES]

    except Exception:
        continue

    low = text.lower()

    for term in SEARCH_TERMS:
        if term.lower() in low or term.lower() in str(p).lower():

            # capture tiny safe context, but never full content.
            idx = low.find(term.lower())

            if idx >= 0:
                start = max(0, idx - 120)
                end = min(len(text), idx + len(term) + 220)

                context = text[start:end]

                # prevent accidental secret-like text dumps
                context = re.sub(
                    r'(?i)(token|secret|password|api[_-]?key)'
                    r'\s*[:=]\s*["\']?[^"\'\s,}]+',
                    r'\1=<REDACTED>',
                    context,
                )

            else:
                context = None

            if len(term_hits[term]) < 30:
                term_hits[term].append({
                    "path": str(p),
                    "context": context,
                })


# ------------------------------------------------------------
# SAFE JSON METADATA EXTRACTION
# ------------------------------------------------------------

SAFE_KEYS = {
    "schema",
    "version",
    "raios_version",
    "system_version",
    "task_id",
    "TASK_ID",
    "run_id",
    "RUN_ID",

    "phase",
    "status",
    "state",
    "verdict",
    "final_verdict",

    "branch",
    "head",
    "head_sha",
    "full_head_sha",
    "commit",
    "commit_sha",

    "created_at",
    "updated_at",
    "timestamp",
    "timestamp_utc",

    "model",
    "model_id",
    "provider",

    "engine",
    "engine_id",
    "runtime_id",

    "GL005_PROVEN",
    "WAL_WRITTEN",
    "EXTRACTED_QWEN_GRANITE",
    "SAFE_TO_REMOVE_SOURCE",

    "PHASE1_COMPLETE",
    "READY_FOR_PHASE2",
}

json_metadata = []

for row in files:
    p = Path(row["absolute_path"])

    if row["secret_like"]:
        continue

    if p.suffix.lower() != ".json":
        continue

    try:
        data = json.loads(
            p.read_text(
                encoding="utf-8",
                errors="strict"
            )
        )
    except Exception:
        continue

    extracted = {}

    def walk(obj, prefix="", depth=0):
        if depth > 6:
            return

        if isinstance(obj, dict):
            for k, v in obj.items():

                if k in SAFE_KEYS:
                    if isinstance(
                        v,
                        (str, int, float, bool, type(None))
                    ):
                        extracted[
                            f"{prefix}{k}"
                        ] = v

                if isinstance(v, (dict, list)):
                    walk(
                        v,
                        prefix=f"{prefix}{k}.",
                        depth=depth + 1,
                    )

        elif isinstance(obj, list):
            for i, item in enumerate(obj[:100]):
                if isinstance(item, (dict, list)):
                    walk(
                        item,
                        prefix=f"{prefix}[{i}].",
                        depth=depth + 1,
                    )

    walk(data)

    if extracted:
        json_metadata.append({
            "path": str(p),
            "sha256": row.get("sha256"),
            "metadata": extracted,
        })


# ------------------------------------------------------------
# ARCHIVE / MODEL / CHECKPOINT CANDIDATES
# ------------------------------------------------------------

archive_exts = {
    ".zip", ".tar", ".gz", ".tgz", ".7z"
}

model_exts = {
    ".gguf", ".bin", ".safetensors", ".pt",
    ".pth", ".onnx", ".ckpt"
}

archives = []
models = []

for row in files:
    ext = row["extension"]

    if ext in archive_exts:
        archives.append(row)

    if ext in model_exts:
        models.append(row)


# ------------------------------------------------------------
# IMPORTANT FILE RANKING
# ------------------------------------------------------------

importance_terms = (
    "manifest",
    "current-state",
    "current_state",
    "state",
    "handoff",
    "decision",
    "architecture",
    "evidence",
    "receipt",
    "checkpoint",
    "raios",
    "brain",
    "cortex",
    "model",
    "learning",
    "skill",
    "gl-002",
    "gl-003",
    "gl-004",
    "gl-005",
    "neurolingua",
    "rif",
)

important = []

for row in files:
    low = (
        row["relative_path"]
        + "/"
        + row["name"]
    ).lower()

    score = sum(
        1 for term in importance_terms
        if term in low
    )

    if score:
        important.append({
            **row,
            "importance_score": score,
        })

important.sort(
    key=lambda x: (
        -x["importance_score"],
        -(x["size"] or 0),
        x["relative_path"]
    )
)


# ------------------------------------------------------------
# REPORT
# ------------------------------------------------------------

report = {
    "schema": "raios.kaggle.asset-lineage.v1",
    "task_id": TASK_ID,
    "generated_at_utc": NOW,

    "notebook": NOTEBOOK_SLUG,

    "evidence_class":
        "LIVE_KAGGLE_RUNTIME_LOCAL_ASSET_INSPECTION",

    "safety": {
        "network": False,
        "gpu": False,
        "tpu": False,
        "model_run": False,
        "git_fetch": False,
        "git_pull": False,
        "git_push": False,
        "delete": False,
        "secret_content_read": False,
    },

    "totals": {
        "files": len(files),
        "input_files": sum(
            1 for x in files
            if x["surface"] == "INPUT"
        ),
        "working_files": sum(
            1 for x in files
            if x["surface"] == "WORKING"
        ),
        "unique_hashes": len(hash_groups),
        "duplicate_hash_groups": len(duplicate_groups),
        "archives": len(archives),
        "model_checkpoint_candidates": len(models),
        "json_metadata_files": len(json_metadata),
    },

    "extensions": dict(extension_counts.most_common()),

    "directories": [
        {
            "path": path,
            "file_count": count,
        }
        for path, count in directory_counts.most_common()
    ],

    "files": files,

    "duplicate_groups": duplicate_groups,

    "term_hits": dict(term_hits),

    "json_safe_metadata": json_metadata,

    "archives": archives,

    "model_checkpoint_candidates": models,

    "important_files_ranked": important[:300],
}

OUT.write_text(
    json.dumps(
        report,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8",
)

report_sha = sha256_file(OUT)

OUT_SHA.write_text(
    f"{report_sha}  {OUT.name}\n",
    encoding="utf-8",
)

# ------------------------------------------------------------
# COMPACT OUTPUT
# ------------------------------------------------------------

print()
print("=" * 80)
print("RAIOS KAGGLE ASSET / LINEAGE MANIFEST")
print("=" * 80)

print(f"NOTEBOOK={NOTEBOOK_SLUG}")
print(
    "EVIDENCE_CLASS="
    "LIVE_KAGGLE_RUNTIME_LOCAL_ASSET_INSPECTION"
)

print(f"FILES_TOTAL={len(files)}")

print(
    "INPUT_FILES="
    f"{sum(1 for x in files if x['surface']=='INPUT')}"
)

print(
    "WORKING_FILES="
    f"{sum(1 for x in files if x['surface']=='WORKING')}"
)

print(
    f"UNIQUE_HASHES={len(hash_groups)}"
)

print(
    f"DUPLICATE_HASH_GROUPS={len(duplicate_groups)}"
)

print(
    f"ARCHIVES={len(archives)}"
)

print(
    f"MODEL_CHECKPOINT_CANDIDATES={len(models)}"
)

print(
    f"JSON_METADATA_FILES={len(json_metadata)}"
)

print()
print("TOP_IMPORTANT_FILES:")

for x in important[:30]:
    print(
        f"  SCORE={x['importance_score']} | "
        f"{x['relative_path']} | "
        f"SIZE={x['size']} | "
        f"SHA={x['sha256']}"
    )

print()
print("VERSION / LINEAGE SIGNALS:")

for term in SEARCH_TERMS:
    hits = term_hits.get(term, [])

    if hits:
        print(
            f"  {term} = {len(hits)} files"
        )

print()
print("SAFE_JSON_METADATA:")

for item in json_metadata[:40]:
    print(
        f"  {item['path']}"
    )

    for k, v in list(
        item["metadata"].items()
    )[:20]:
        print(
            f"      {k}={v}"
        )

print()
print(
    f"REPORT_JSON={OUT}"
)

print(
    f"REPORT_SHA256={report_sha}"
)

print(
    f"HASH_SIDECAR={OUT_SHA}"
)

print()
print(
    "FINAL_VERDICT="
    "KAGGLE_ASSET_LINEAGE_READY_FOR_C3_RECONCILIATION"
)

print("=" * 80)


RAIOS KAGGLE ASSET / LINEAGE MANIFEST
NOTEBOOK=greenylife/notebook8d96bf4e18
EVIDENCE_CLASS=LIVE_KAGGLE_RUNTIME_LOCAL_ASSET_INSPECTION
FILES_TOTAL=128
INPUT_FILES=124
WORKING_FILES=4
UNIQUE_HASHES=127
DUPLICATE_HASH_GROUPS=1
ARCHIVES=0
MODEL_CHECKPOINT_CANDIDATES=0
JSON_METADATA_FILES=120

TOP_IMPORTANT_FILES:
  SCORE=4 | datasets/greenylife/raios-cognitive-state/RAIOS-STATE-LATEST/RAIOS/raios-cognitive-factory/state/manifests/learning-ledger.json | SIZE=597 | SHA=559904ef8138a503cb0ad1b224014818e79f98ce956e8ac54faf7371dc73ceb5
  SCORE=3 | datasets/greenylife/raios-cognitive-state/RAIOS-STATE-LATEST/RAIOS/DURABLE-MANIFEST.json | SIZE=28836 | SHA=a271723d9b35ede43f4c16da07b543a57583d85a38ad946012992dee3f946015
  SCORE=3 | datasets/greenylife/raios-cognitive-state/RAIOS-STATE-LATEST/RAIOS/raios-cognitive-factory/state/evidence/e662b7f7c6416cee22681f02bd4859c3945019b0f391c6ace057ab2d4ca85439.json | SIZE=14862 | SHA=bb082d6e0bad130f42fa45ab564dacc59fc795be7b8cfa0d88a7eaa3897bfe72
  SCORE=